### standard

In [ ]:
# Load necessary libraries
import sys
import os
import anndata as ad

import pandas as pd
import collections
import scanpy as sc

import seaborn as sns; sns.set(color_codes=True)
import matplotlib.pyplot as plt
import numpy as np

import warnings
warnings.filterwarnings('ignore')

def custom_scaled_heatmap(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean', replace_na = False):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure()
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
        
        
    matrix_data_scaled = matrix_data_bulked/(np.max(matrix_data_bulked, axis=0))
    
    if replace_na:
        matrix_data_scaled = matrix_data_scaled.fillna(0)
    
    sns.heatmap(matrix_data_scaled.transpose(), square = True, linewidths=0.01, linecolor='black', cmap="inferno")
    plt.xlabel(None)
    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()
        
        

        
def custom_scaled_heatmap_Z(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean', replace_na = False):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure()
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    
    # apply z score transform
    #matrix_data = (matrix_data - np.mean(matrix_data))/np.std(matrix_data)

    
    matrix_data[group_by] = adata.obs[group_by]
    
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.mean(matrix_data_bulked))/np.std(matrix_data_bulked)
    matrix_data_scaled = matrix_data_scaled.clip(lower=-2, upper=2)
    
    if replace_na:
        matrix_data_scaled = matrix_data_scaled.fillna(np.nanmin(matrix_data_scaled))
    
    
    
    sns.heatmap(matrix_data_scaled.transpose(), square = True, linewidths=0.01, linecolor='black', cmap="inferno")
    plt.xlabel(None)
    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()
        
        
def custom_scaled_heatmap_minmax(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean', replace_na = False):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure()
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    
    # apply z score transform
    #matrix_data = (matrix_data - np.mean(matrix_data))/np.std(matrix_data)

    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.min(matrix_data_bulked))/(np.max(matrix_data_bulked) - np.min(matrix_data_bulked))
    
    if replace_na:
        matrix_data_scaled = matrix_data_scaled.fillna(0)
    
    sns.heatmap(matrix_data_scaled.transpose(), square = True, linewidths=0.01, linecolor='black', cmap="inferno")
    plt.xlabel(None)
    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()

In [ ]:


def custom_scaled_heatmap_dot(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = matrix_data_bulked/(np.max(matrix_data_bulked, axis=0))
    
    
    data = matrix_data_scaled
    #bubble_sizes = 100
    
    colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
    
    # Create a figure and axis
    fig, ax = plt.subplots()

    # Loop through the DataFrame and create bubbles
    for i, row in enumerate(data.index):
        for j, col in enumerate(data.columns):
            value = data.at[row, col]  # Use DataFrame value for color
            size = 100  # Use additional value for bubble size
            color = cmap(value)  # Use DataFrame value for color
            ax.scatter(j, i, c=[color], s=size, alpha=0.7)
    print(plt.cm.Blues(value))
    
    ax.set_xticks(np.arange(len(data.columns)))
    ax.set_yticks(np.arange(len(data.index)))
    ax.set_xticklabels(data.columns, rotation=90)
    ax.set_yticklabels(data.index)
    

    # Add grid lines
    ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

    plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


    ax.grid(which="minor", color="black", linestyle='-', linewidth=2)
    
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()


        
        
        
        
        





def custom_scaled_heatmap_Z_dot(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.mean(matrix_data_bulked))/np.std(matrix_data_bulked)
    matrix_data_scaled = matrix_data_scaled.clip(lower=-2, upper=2)
    
    data = matrix_data_scaled
    #bubble_sizes = 100
    
    colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
    
    # Create a figure and axis
    fig, ax = plt.subplots()

    # Loop through the DataFrame and create bubbles
    for i, row in enumerate(data.index):
        for j, col in enumerate(data.columns):
            value = data.at[row, col]  # Use DataFrame value for color
            size = 100  # Use additional value for bubble size
            color = cmap(value)  # Use DataFrame value for color
            ax.scatter(j, i, c=[color], s=size, alpha=0.7)
    print(plt.cm.Blues(value))
    
    ax.set_xticks(np.arange(len(data.columns)))
    ax.set_yticks(np.arange(len(data.index)))
    ax.set_xticklabels(data.columns, rotation=90)
    ax.set_yticklabels(data.index)
    

    # Add grid lines
    ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

    plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


    ax.grid(which="minor", color="black", linestyle='-', linewidth=2)
    
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()



def custom_scaled_heatmap_minmax_dot(adata, layer = 'unscaled_data', group_by = 'leiden', saveto = False, var_names = False, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.min(matrix_data_bulked))/(np.max(matrix_data_bulked) - np.min(matrix_data_bulked))
    
    
    data = matrix_data_scaled
    #bubble_sizes = 100
    
    colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
    
    # Create a figure and axis
    fig, ax = plt.subplots()

    # Loop through the DataFrame and create bubbles
    for i, row in enumerate(data.index):
        for j, col in enumerate(data.columns):
            value = data.at[row, col]  # Use DataFrame value for color
            size = 100  # Use additional value for bubble size
            color = cmap(value)  # Use DataFrame value for color
            ax.scatter(j, i, c=[color], s=size, alpha=0.7)
    print(plt.cm.Blues(value))
    
    ax.set_xticks(np.arange(len(data.columns)))
    ax.set_yticks(np.arange(len(data.index)))
    ax.set_xticklabels(data.columns, rotation=90)
    ax.set_yticklabels(data.index)
    

    # Add grid lines
    ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

    plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


    ax.grid(which="minor", color="black", linestyle='-', linewidth=2)
    
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()



In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial")

sns.set_style('white')

In [ ]:
#adataScaled object for iba1 cels (must have cells with scaled and batch corrected gene expression values)
adataScaled = sc.read_h5ad('final_h5ads/adataScaled_prior_to_clustering.h5')

# import adata object with proper cluster annotations
clustering_adata = sc.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')

# add original unscaled data to adatascaled object as a new slot

adataScaled.layers['unscaled_data'] = clustering_adata.to_df()[adataScaled.var_names]

# add cluster annotations to adatascaled object
clusterdict = dict(zip(clustering_adata.obs.ImageID_CellID, clustering_adata.obs.leiden))
adataScaled.obs['leiden'] = [clusterdict[i] for i in adataScaled.obs.ImageID_CellID]                    

# abetadistancedict = dict(zip(clustering_adata.obs.ImageID_CellID, clustering_adata.obs.distance_to_nearest_aBeta))
# adataScaled.obs['distance_to_nearest_aBeta'] = [abetadistancedict[i] for i in adataScaled.obs.ImageID_CellID]          

# vasculaturedistancedict = dict(zip(clustering_adata.obs.ImageID_CellID, clustering_adata.obs.distance_to_nearest_vasculature))
# adataScaled.obs['distance_to_nearest_vasculature'] = [vasculaturedistancedict[i] for i in adataScaled.obs.ImageID_CellID]  
                
# apoedistancedict = dict(zip(clustering_adata.obs.ImageID_CellID, clustering_adata.obs.distance_to_nearest_ApoE))
# adataScaled.obs['distance_to_nearest_ApoE'] = [apoedistancedict[i] for i in adataScaled.obs.ImageID_CellID]    
    
pre_sub_leidendict = dict(zip(clustering_adata.obs.ImageID_CellID, clustering_adata.obs.pre_subclustering_leiden))
adataScaled.obs['pre_subclustering_leiden'] = [pre_sub_leidendict[i] for i in adataScaled.obs.ImageID_CellID]

adataScaled.obs['pre_subclustering_leiden'] = adataScaled.obs['pre_subclustering_leiden'].astype('category')

change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

# cMapDict = {'MG_0': '#1f77b4',
#  'MG_1': '#ff7f0e',
#  'MO_MAC_0': '#279e68',
#  'MO_MAC_1': '#d62728',
#  'MO_MAC_2': '#aa40fc',
#  'default': '#8c564b'
# }

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}

# cMapDict = {0: '#1f77b4',
#  1: '#ff7f0e',
#  2: '#279e68',
#  3: '#d62728',
#  4: '#aa40fc',
#  5: '#8c564b',
#  6: '#e377c2',
#  7: '#b5bd61',
#  8: '#17becf',
#  9: '#aec7e8',
#  10: '#ffbb78'}






cMapDict_presub = {
0:'#1f77b4',
1:'#279e68',
2:'gray'
}

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']

colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))

adataScaled.obs['leiden'] = [change_cluster_names_dict[i] for i in adataScaled.obs['leiden']]

adataScaled.obs['leiden'] = pd.Categorical(adataScaled.obs['leiden'], categories=new_name_order, ordered=False)



sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='pre_subclustering_leiden',s=1,palette=cMapDict_presub)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/pre_subclustering_leiden_tsne.tiff')
plt.savefig('figures/pre_subclustering_leiden_tsne.svg', format = 'svg')






g = sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='pre_subclustering_leiden',s=1,palette=cMapDict_presub, legend=False)

plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/pre_subclustering_leiden_tsne_nolegend.tiff')
plt.savefig('figures/pre_subclustering_leiden_tsne_nolegend.svg', format = 'svg')




# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True, layer='unscaled_data')

# fig.savefig('figures/heatmap_presubleiden.tiff')
# fig.savefig('figures/heatmap_presubleiden.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden.tiff')
custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden.svg')










adataScaled = adataScaled[adataScaled.obs.leiden != 'default']


In [ ]:



from scipy import stats

from matplotlib.colors import LinearSegmentedColormap


# Define custom color palette scaling from white to blue
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

sc.pl.tsne(clustering_adata, color=adataScaled.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False)
plt.savefig('figures/marker_feature_plot_bluewhite_noscaling.tiff')
plt.savefig('figures/marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')


# also show additional markers

additional_markers = ['GFAP', 'NeuN', 'Claudin5']

sc.pl.tsne(clustering_adata, color=additional_markers,use_raw=False, return_fig = True, color_map=cmap)
sc.pl.tsne(clustering_adata, color=additional_markers,use_raw=False, return_fig = True, color_map=cmap)




only_BLM = clustering_adata[clustering_adata.obs.leiden == 'MO_MAC_2']
only_BLM_no_AML = clustering_adata[(clustering_adata.obs.leiden == 'MO_MAC_2') & (clustering_adata.obs.ImageIDOLD != '3628')]
only_BLM_only_CTL = clustering_adata[(clustering_adata.obs.leiden == 'MO_MAC_2') & (clustering_adata.obs.ImageType == 'Ctl')]
only_BLM_only_AD_and_AML = clustering_adata[(clustering_adata.obs.leiden == 'MO_MAC_2') & ((clustering_adata.obs.ImageType == 'AD') | (clustering_adata.obs.ImageIDOLD == '3628'))]


expression_df = only_BLM.to_df()
expression_df['sample'] = only_BLM.obs.ImageIDOLD
expression_df_bulked = expression_df.groupby('sample').mean()


features_to_profile = ['TMEM119', 'CD163']

for feature in features_to_profile:
    
    sc.pl.violin(only_BLM, feature, groupby = 'ImageIDOLD', stripplot=True)
    sc.pl.violin(only_BLM, feature, groupby = 'ImageType', stripplot=True)
    
    
    # first, test between AD and control
    
    AD_levels = only_BLM[only_BLM.obs.ImageType == 'AD'].to_df()[feature]
    control_levels = only_BLM[only_BLM.obs.ImageType == 'Ctl'].to_df()[feature]
    control_levels_minus_AML = only_BLM[(only_BLM.obs.ImageType == 'Ctl') & (only_BLM.obs.ImageIDOLD != '3628')].to_df()[feature]
    only_AML = only_BLM[(only_BLM.obs.ImageType == 'Ctl') & (only_BLM.obs.ImageIDOLD == '3628')].to_df()[feature]

    print('--------------------------------------------------------------------------------------------------------------')
    print('t statistic and p value for t test between AD and control ' + feature + ' with all samples included')

    t_statistic, t_p_value = stats.ttest_ind(AD_levels, control_levels)
    
    print(t_p_value)
    print(t_statistic)

    
    print('--------------------------------------------------------------------------------------------------------------')
    
#     print('--------------------------------------------------------------------------------------------------------------')
#     print('u statistic and p value for mann whitney u test between AD and control ' + feature + ' with all samples included')

#     u_statistic, u_p_value = stats.mannwhitneyu(AD_levels, control_levels)
    
#     print(u_p_value)
#     print(u_statistic)
    
#     print('--------------------------------------------------------------------------------------------------------------')
    
    
    
    print('--------------------------------------------------------------------------------------------------------------')
    print('t statistic and p value for t test between AD and control ' + feature + ' with all samples included BULKED')

    AD_levels_bulked = expression_df_bulked[expression_df_bulked.index.isin(['2997', '3155', '3196', '3280'])][feature]
    CONTROL_levels_bulked = expression_df_bulked[expression_df_bulked.index.isin(['1796', '1873', '3026', '3628'])][feature]
    t_statistic, t_p_value = stats.ttest_ind(AD_levels_bulked, CONTROL_levels_bulked)
    
    print(t_p_value)
    print(t_statistic)
    
    print('--------------------------------------------------------------------------------------------------------------')
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AD sample means': AD_levels_bulked, 'CONTROL sample means': CONTROL_levels_bulked})

    # Create box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.swarmplot(data=df, color="black", alpha=0.5)  # Overlay points
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/bulked_expression_of_' + feature + '_ad_vs_control_all_samples_box_and_whisker.tiff')
    plt.savefig('figures/bulked_expression_of_' + feature + '_ad_vs_control_all_samples_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AD': AD_levels, 'CONTROL': control_levels})

    # create sc box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.stripplot(data = df, size = 1, color='black')
    #sns.swarmplot(data=df, color="black", alpha=0.5, dodge=0.1)  # Overlay points
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_all_samples_box_and_whisker.tiff')
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_all_samples_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AD': AD_levels, 'CONTROL (minus AML)': control_levels_minus_AML, 'AML':only_AML})

    # create sc box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5, whis=np.Inf)
    sns.stripplot(data = df, size = 1, color='black')
    #sns.swarmplot(data=df, color="black", alpha=0.5, dodge=0.1)  # Overlay points
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_vs_AML_box_and_whisker.tiff')
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_vs_AML_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    
    
    
    
    
    plt.figure(figsize=(8, 6))
    
    data1 = AD_levels
    data2 = control_levels
        
    plt.hist(data1, bins=30, alpha=0.5, color='blue', label='AD')
    plt.hist(data2, bins=30, alpha=0.5, color='red', label='control')

    axes = plt.gca()

    ylim = axes.get_ylim()[1]

    plt.vlines(x = np.mean(data1), colors = 'blue', ymin = 0, ymax = ylim)
    plt.vlines(x = np.mean(data2), colors = 'red', ymin = 0, ymax = ylim)



    plt.xlabel('Expression')
    plt.ylabel('Frequency')
    plt.title(feature + ' (ALL SAMPLES INCLUDED)')
    plt.legend()
    plt.grid(True)
    
    
    # next, test between AD and control with AML gone
    
    AD_levels = only_BLM_no_AML[only_BLM_no_AML.obs.ImageType == 'AD'].to_df()[feature]
    control_levels = only_BLM_no_AML[only_BLM_no_AML.obs.ImageType == 'Ctl'].to_df()[feature]
    
    print('t statistic and p value for t test between AD and control ' + feature + ' with AML excluded')
    
    t_statistic, t_p_value = stats.ttest_ind(AD_levels, control_levels)
    
    print(t_p_value)
    print(t_statistic)
    print('--------------------------------------------------------------------------------------------------------------')
    
    print('--------------------------------------------------------------------------------------------------------------')
    print('t statistic and p value for t test between AD and control ' + feature + ' with AML excluded BULKED')

    AD_levels_bulked = expression_df_bulked[expression_df_bulked.index.isin(['2997', '3155', '3196', '3280'])][feature]
    CONTROL_levels_bulked = expression_df_bulked[expression_df_bulked.index.isin(['1796', '1873', '3026'])][feature]
    t_statistic, t_p_value = stats.ttest_ind(AD_levels_bulked, CONTROL_levels_bulked)
    
    print(t_p_value)
    print(t_statistic)
    
    print('--------------------------------------------------------------------------------------------------------------')
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AD sample means': AD_levels_bulked, 'CONTROL sample means': CONTROL_levels_bulked})

    # Create box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.swarmplot(data=df, color="black", alpha=0.5)  # Overlay points
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/bulked_expression_of_' + feature + '_ad_vs_control_AML_exluded_box_and_whisker.tiff')
    plt.savefig('figures/bulked_expression_of_' + feature + '_ad_vs_control_AML_exluded_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AD': AD_levels, 'CONTROL (excluding AML)': control_levels})

    # create sc box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.stripplot(data = df, size = 1, color='black')
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_AML_excluded_box_and_whisker.tiff')
    plt.savefig('figures/sc_expression_of_' + feature + '_ad_vs_control_AML_excluded_box_and_whisker.svg', format = 'svg')
    plt.show()

    
    plt.figure(figsize=(8, 6))
    
    data1 = AD_levels
    data2 = control_levels
        
    plt.hist(data1, bins=30, alpha=0.5, color='blue', label='AD')
    plt.hist(data2, bins=30, alpha=0.5, color='red', label='control (minus AML)')

    axes = plt.gca()

    ylim = axes.get_ylim()[1]

    plt.vlines(x = np.mean(data1), colors = 'blue', ymin = 0, ymax = ylim)
    plt.vlines(x = np.mean(data2), colors = 'red', ymin = 0, ymax = ylim)



    plt.xlabel('Expression')
    plt.ylabel('Frequency')
    plt.title(feature + ' (AML EXCLUDED FROM CTL)')
    plt.legend()
    plt.grid(True)
    
    
    
    
    # next, test between AML and other control
    
    AML_levels = only_BLM_only_CTL[only_BLM_only_CTL.obs.ImageIDOLD == '3628'].to_df()[feature]
    control_levels = only_BLM_only_CTL[only_BLM_only_CTL.obs.ImageIDOLD != '3628'].to_df()[feature]
    
    print('t statistic and p value for t test between AML and control ' + feature)
    
    t_statistic, t_p_value = stats.ttest_ind(AML_levels, control_levels)
    
    print(t_p_value)
    print(t_statistic)
    print('--------------------------------------------------------------------------------------------------------------')

    
    plt.figure(figsize=(8, 6))
    
    data1 = AML_levels
    data2 = control_levels
        
    plt.hist(data1, bins=30, alpha=0.5, color='blue', label='AML')
    plt.hist(data2, bins=30, alpha=0.5, color='red', label='control (minus AML)')

    axes = plt.gca()

    ylim = axes.get_ylim()[1]

    plt.vlines(x = np.mean(data1), colors = 'blue', ymin = 0, ymax = ylim)
    plt.vlines(x = np.mean(data2), colors = 'red', ymin = 0, ymax = ylim)



    plt.xlabel('Expression')
    plt.ylabel('Frequency')
    plt.title(feature + ' (AML vs CTL)')
    plt.legend()
    plt.grid(True)
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AML': AML_levels, 'CONTROL (excluding AML)': control_levels})

    # create sc box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.stripplot(data = df, size = 1, color='black')
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/sc_expression_of_' + feature + '_AML_vs_control_box_and_whisker.tiff')
    plt.savefig('figures/sc_expression_of_' + feature + '_AML_vs_control_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    
    
    
    # finally, test between AML and AD
    
    AML_levels = only_BLM_only_AD_and_AML[only_BLM_only_AD_and_AML.obs.ImageIDOLD == '3628'].to_df()[feature]
    AD_levels = only_BLM_only_AD_and_AML[only_BLM_only_AD_and_AML.obs.ImageIDOLD != '3628'].to_df()[feature]
    
    print('t statistic and p value for t test between AML and AD ' + feature)
    
    t_statistic, t_p_value = stats.ttest_ind(AML_levels, AD_levels)
    
    print(t_p_value)
    print(t_statistic)
    print('--------------------------------------------------------------------------------------------------------------')

    
    plt.figure(figsize=(8, 6))
    
    data1 = AML_levels
    data2 = AD_levels
        
    plt.hist(data1, bins=30, alpha=0.5, color='blue', label='AML')
    plt.hist(data2, bins=30, alpha=0.5, color='red', label='AD')

    axes = plt.gca()

    ylim = axes.get_ylim()[1]

    plt.vlines(x = np.mean(data1), colors = 'blue', ymin = 0, ymax = ylim)
    plt.vlines(x = np.mean(data2), colors = 'red', ymin = 0, ymax = ylim)



    plt.xlabel('Expression')
    plt.ylabel('Frequency')
    plt.title(feature + ' (AML vs AD)')
    plt.legend()
    plt.grid(True)
    
    
    # Combine data into a DataFrame
    df = pd.DataFrame({'AML': AML_levels, 'AD': AD_levels})

    # create sc box and whisker plot
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, width=0.5)
    sns.stripplot(data = df, size = 1, color='black')
    plt.title('Box and Whisker Plot')
    plt.ylabel(feature)
    plt.savefig('figures/sc_expression_of_' + feature + '_AML_vs_AD_box_and_whisker.tiff')
    plt.savefig('figures/sc_expression_of_' + feature + '_AML_vs_AD_box_and_whisker.svg', format = 'svg')
    plt.show()
    
    
    
    
    
    
    
    
    
    
    
    
    



In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cadata = clustering_adata[clustering_adata.obs['Parent'].isin(['Grey Matter', 'White Matter'])]
cadata = cadata[cadata.obs.leiden != 'default']

# Define custom color palette scaling from white to blue
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

sc.pl.tsne(cadata, color=adataScaled.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False)
plt.savefig('figures/marker_feature_plot_bluewhite_noscaling_nodefault.tiff')
plt.savefig('figures/marker_feature_plot_bluewhite_noscaling_nodefault.svg', format = 'svg')

In [ ]:
# remove any meningeal macrophages

adataScaled = adataScaled[adataScaled.obs['Parent'].isin(['Grey Matter', 'White Matter'])]



In [ ]:
fig = sc.pl.tsne(adataScaled, color=adataScaled.var_names,use_raw=False, return_fig = True, layer='unscaled_data')
fig.savefig('figures/marker_tsne.tiff')
fig.savefig('figures/marker_tsne.svg', format = 'svg')



In [ ]:


colors = list(cMapDict.values())
leiden_clusters = list(set(adataScaled.obs['leiden']))



custom_key = lambda x: new_name_order.index(x)

leiden_clusters = sorted(leiden_clusters, key=custom_key)


cMapDict = {}

for i in zip(colors, leiden_clusters):
    print(i)
    cMapDict[i[1]] = i[0]


regiondict = {'Grey Matter':'GM', 'White Matter':'WM', np.nan: 'not_annotated'}
adataScaled.obs['Region'] = [regiondict[i] for i in adataScaled.obs['Parent']]


In [ ]:

#sc.pl.tsne(adataScaled, color=['leiden','ImageTypev2','ImageType'], size=2,ncols=3,sort_order=False, save='_umapsV4.png')
#fig,ax=plt.subplots(1,4, figsize=(13,3.5))

# fig = plt.figure()
# sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
#                 hue='broad_leiden',s=1,)


# plt.legend(ncol=4,loc=(0,-0.4))
# plt.xlabel('tSNE1')
# plt.ylabel('tSNE2')
# plt.tight_layout()

# plt.savefig('pixel_clustering_newmethod/paper_images/broad_leiden_clustering_tsne.tiff')

# fig = plt.figure()



fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='leiden', palette=cMapDict,s=1,)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/leiden_clustering_tsne.tiff')
plt.savefig('figures/leiden_clustering_tsne.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='pre_subclustering_leiden',s=1,palette=cMapDict_presub)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault.tiff')
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='Region',s=1, palette=['tab:blue','tab:pink'], hue_order=['GM','WM'])


plt.legend(ncol=4,loc=(0,-0.3))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/GM_WM_tsne.tiff')
plt.savefig('figures/GM_WM_tsne.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='ImageType',s=1,palette=sc.pl.palettes.godsnot_102[3:5], hue_order=['AD','Ctl'])

plt.legend(ncol=4,loc=(0,-0.3))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/AD_CNT_tsne.tiff')
plt.savefig('figures/AD_CNT_tsne.svg', format = 'svg')



fig = plt.figure()

sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='ImageID', palette=sc.pl.palettes.godsnot_102[9:17],s=1,  hue_order=sorted(list(adataScaled.obs['ImageID'].unique()))
)
plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/sample_tsne.tiff')
plt.savefig('figures/sample_tsne.svg', format = 'svg')













fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='leiden', palette=cMapDict,s=1,legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/leiden_clustering_tsne_nolegend.tiff')
plt.savefig('figures/leiden_clustering_tsne_nolegend.svg', format = 'svg')



fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='pre_subclustering_leiden',s=1,palette=cMapDict_presub, legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault_nolegend.tiff')
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault_nolegend.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='Region',s=1, palette=['tab:blue','tab:pink'], hue_order=['GM','WM'], legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/GM_WM_tsne_nolegend.tiff')
plt.savefig('figures/GM_WM_tsne_nolegend.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='ImageType',s=1,palette=sc.pl.palettes.godsnot_102[3:5], hue_order=['AD','Ctl'], legend=False)

plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/AD_CNT_tsne_nolegend.tiff')
plt.savefig('figures/AD_CNT_tsne_nolegend.svg')



fig = plt.figure()

sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
                hue='ImageID', palette=sc.pl.palettes.godsnot_102[9:17],s=1, legend=False, hue_order=sorted(list(adataScaled.obs['ImageID'].unique()))
)
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/sample_tsne_nolegend.tiff')
plt.savefig('figures/sample_tsne_nolegend.svg')



















In [ ]:

sc.pl.scatter(adataScaled, basis = 'tsne', color='leiden', palette=cMapDict.values(), legend_loc = 'none', show=False, size=4, title = '')
plt.savefig('figures/leiden_clustering_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/leiden_clustering_tsne_nolegend_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(adataScaled, basis = 'tsne', color='pre_subclustering_leiden', palette=cMapDict_presub.values(), legend_loc = 'none', show=False, size=4, title = '')
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault_nolegend_EASYEDIT.tiff')
plt.savefig('figures/pre_subclustering_leiden_tsne_nodefault_nolegend_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(adataScaled, basis = 'tsne', color='Region', palette={'GM':'tab:blue','WM':'tab:pink'}.values(), legend_loc = 'none',  show=False, size=4, title = '')
plt.savefig('figures/GM_WM_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/GM_WM_tsne_nolegend_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(adataScaled, basis = 'tsne', color='ImageType', palette=sc.pl.palettes.godsnot_102[3:5], legend_loc = 'none',  show=False, size=4, title = '')
plt.savefig('figures/AD_CNT_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/AD_CNT_tsne_nolegend_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(adataScaled, basis = 'tsne', color='ImageID', palette=sc.pl.palettes.godsnot_102[9:17], legend_loc = 'none',  show=False, size=4, title = '')
plt.savefig('figures/sample_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/sample_tsne_nolegend_EASYEDIT.svg', format = 'svg')




In [ ]:
# Plot heatmap
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='broad_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None, return_fig=True,swap_axes=True, vmin=-0.50,vmax=0.75)

# fig.savefig('pixel_clustering_newmethod/paper_images/broad_heatmap.tiff')


# sc.tl.dendrogram(adataScaled, groupby='leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden.tiff')
# fig.savefig('figures/heatmap_leiden.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden.tiff')
custom_scaled_heatmap(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM.tiff')
custom_scaled_heatmap(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM.tiff')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault.tiff')
custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault.svg')


In [ ]:
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_Z.tiff')
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_Z.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_Z.tiff')
custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_Z.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_Z.tiff')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_Z.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_Z.tiff')
custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_Z.svg')

In [ ]:
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_minmax.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_minmax.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_minmax.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_minmax.svg')




In [ ]:
# Plot heatmap
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='broad_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None, return_fig=True,swap_axes=True, vmin=-0.50,vmax=0.75)

# fig.savefig('pixel_clustering_newmethod/paper_images/broad_heatmap.tiff')


# sc.tl.dendrogram(adataScaled, groupby='leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden.tiff')
# fig.savefig('figures/heatmap_leiden.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_median.tiff', style='median')
custom_scaled_heatmap(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_median.tiff', style='median')
custom_scaled_heatmap(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_median.tiff', style='median')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_median.svg', style='median')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_median.tiff', style='median')
custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_median.svg', style='median')


In [ ]:
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_Z_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_Z_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_Z_median.svg', style='median')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_Z_median.svg', style='median')

In [ ]:
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', saveto='figures/heatmap_leiden_minmax_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', saveto='figures/heatmap_leiden_GM_minmax_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', saveto='figures/heatmap_leiden_WM_minmax_median.svg', style='median')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', saveto='figures/heatmap_presubleiden_nodefault_minmax_median.svg', style='median')




In [ ]:
# Plot heatmap
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='broad_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None, return_fig=True,swap_axes=True, vmin=-0.50,vmax=0.75)

# fig.savefig('pixel_clustering_newmethod/paper_images/broad_heatmap.tiff')


# sc.tl.dendrogram(adataScaled, groupby='leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden.tiff')
# fig.savefig('figures/heatmap_leiden.svg', format = 'svg')

morph_feats = ['area µm²', 'perimeter_um', 'cell_circularity', 'cell_solidity',
       'soma_area_um2', 'soma_circularity', 'number_of_processes',
       'number_of_branches', 'branching_degree', 'average_process_length_um',
       'average_process_thickness_um', 'bi_um']

custom_scaled_heatmap(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats.tiff')
custom_scaled_heatmap(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_GM.tiff')
# fig.savefig('figures/heatmap_leiden_GM.svg')

custom_scaled_heatmap(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM.tiff')
custom_scaled_heatmap(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_WM.tiff')
# fig.savefig('figures/heatmap_leiden_WM.svg')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM.tiff')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_nodefault.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault.svg')


In [ ]:
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_Z.tiff')
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_Z.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_GM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_GM.svg')

custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_Z.tiff')
custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_Z.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_WM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_WM.svg')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_Z.tiff')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_Z.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.svg', format = 'svg')

custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_Z.tiff')
custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_Z.svg')

In [ ]:
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_minmax.svg')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_GM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_GM.svg')

custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_minmax.svg')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_WM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_WM.svg')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_minmax.svg')


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.svg', format = 'svg')

custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_minmax.tiff')
custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_minmax.svg')




In [ ]:
# Plot heatmap
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='broad_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None, return_fig=True,swap_axes=True, vmin=-0.50,vmax=0.75)

# fig.savefig('pixel_clustering_newmethod/paper_images/broad_heatmap.tiff')


# sc.tl.dendrogram(adataScaled, groupby='leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden.tiff')
# fig.savefig('figures/heatmap_leiden.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_median.tiff', style='median')
custom_scaled_heatmap(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_GM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_GM.svg')

custom_scaled_heatmap(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_median.tiff', style='median')
custom_scaled_heatmap(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_WM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_WM.svg')
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_median.tiff', style='median', replace_na=True)
custom_scaled_heatmap(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_median.svg', style='median', replace_na=True)


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.svg', format = 'svg')

custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_median.tiff', style='median')
custom_scaled_heatmap(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_median.svg', style='median')


In [ ]:
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_Z_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_GM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_GM.svg')

custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_Z_median.tiff', style='median')
custom_scaled_heatmap_Z(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_Z_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_WM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_WM.svg')
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_Z_median.tiff', style='median', replace_na=True)
custom_scaled_heatmap_Z(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_Z_median.svg', style='median', replace_na=True)


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.svg', format = 'svg')

custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_Z_median.tiff', style='median', replace_na = True)
custom_scaled_heatmap_Z(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_Z_median.svg', style='median', replace_na = True)

In [ ]:
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_minmax_median.svg', style='median')






adataScaled_GM = adataScaled[adataScaled.obs['Parent'] == 'Grey Matter']
# fig = sc.pl.matrixplot(adataScaled_GM, var_names = adataScaled_GM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_GM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_GM.svg')

custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_minmax_median.tiff', style='median')
custom_scaled_heatmap_minmax(adataScaled_GM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_GM_minmax_median.svg', style='median')



adataScaled_WM = adataScaled[adataScaled.obs['Parent'] == 'White Matter']
# fig = sc.pl.matrixplot(adataScaled_WM, var_names = adataScaled_WM.var_names,groupby='leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_leiden_morph_feats_WM.tiff')
# fig.savefig('figures/heatmap_leiden_morph_feats_WM.svg')
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_minmax_median.tiff', style='median', replace_na=True)
custom_scaled_heatmap_minmax(adataScaled_WM, group_by='leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_leiden_morph_feats_WM_minmax_median.svg', style='median', replace_na=True)


# sc.tl.dendrogram(adataScaled, groupby='pre_subclustering_leiden')
# fig = sc.pl.matrixplot(adataScaled, var_names = adataScaled.var_names,groupby='pre_subclustering_leiden', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.tiff')
# fig.savefig('figures/heatmap_presubleiden_morph_feats_nodefault.svg', format = 'svg')

custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_minmax_median.tiff', style='median', replace_na=True)
custom_scaled_heatmap_minmax(adataScaled, group_by='pre_subclustering_leiden', layer='meta', var_names=morph_feats, saveto='figures/heatmap_presubleiden_morph_feats_nodefault_minmax_median.svg', style='median', replace_na=True)




In [ ]:
## plot spatila maps
i = 0
j = 0
fig,ax=plt.subplots(2,4,figsize=(16,10))

for im in adataScaled.obs.ImageID.unique():
    ss = adataScaled[(adataScaled.obs.ImageID.isin([im]))]#sc.pp.subsample(adataScaled[adataScaled.obs.leiden!=10],fraction=0.8, copy=True)
    
    ss.uns['leiden_colors']=[cMapDict[x] for x in ss.obs.leiden.unique().categories]
    print(im, [x for x in sorted(ss.obs.leiden.unique())],len(ss.uns['leiden_colors']))

    if ss.obs.ImageType.values[0] == 'Ctl':
        if i<3:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False, color_map=cMapDict)
        i+=1
    else:
        if j<4:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[1][j],size=12.0,title=im,alpha=0.9,
                          show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[1][j],size=12.0,title=im,alpha=0.9, show=False)
        j+=1
plt.tight_layout()
plt.savefig('figures/microglia_clusters_dotplot.tiff')
plt.savefig('figures/microglia_clusters_dotplot.svg', format = 'svg')



In [ ]:
# plot spatila maps vert
i = 0
j = 0
fig,ax=plt.subplots(4,2,figsize=(10,16))

for im in adataScaled.obs.ImageID.unique():
    ss = adataScaled[(adataScaled.obs.ImageID.isin([im]))]#sc.pp.subsample(adataScaled[adataScaled.obs.leiden!=10],fraction=0.8, copy=True)
    
    ss.uns['leiden_colors']=[cMapDict[x] for x in ss.obs.leiden.unique().categories]
    print(im, [x for x in sorted(ss.obs.leiden.unique())],len(ss.uns['leiden_colors']))

    if ss.obs.ImageType.values[0] == 'Ctl':
        if i<4:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[i][0],size=12.0,title=im,alpha=0.9, 
                      show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[i][0],size=12.0,title=im,alpha=0.9, 
                      show=False, color_map=cMapDict)
        i+=1
    else:
        if j<1:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[j][1],size=12.0,title=im,alpha=0.9,
                          show=False)
        else:
            sc.pl.scatter(ss, basis='spatial',color=['leiden'],ax=ax[j][1],size=12.0,title=im,alpha=0.9, show=False,legend_loc='none')
        j+=1
plt.tight_layout()
plt.savefig('figures/microglia_clusters_dotplot_vertical.tiff')
plt.savefig('figures/microglia_clusters_dotplot_vertical.svg', format = 'svg')



In [ ]:
# plot regions
# plot spatila maps
i = 0
j = 0
fig,ax=plt.subplots(2,4,figsize=(16,10))

for im in adataScaled.obs.ImageID.unique():
    ss = adataScaled[(adataScaled.obs.ImageID.isin([im]))]
    cDict = {'GM':'tab:blue','WM':'tab:pink', 'not_annotated':'gray'}
    ss.uns['Region_colors']=[cDict[x] for x in sorted(ss.obs.Region.unique())]
    if ss.obs.ImageType.values[0] == 'Ctl':
        if i<3:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False,legend_loc='none',)
        else:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False,)
        i+=1
    else:
        if j<4:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[1][j],size=12.0,title=im,alpha=0.9,
                          show=False,legend_loc='none',)
        else:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[1][j],size=12.0,title=im,alpha=0.9,
                          show=False,)
        j+=1
plt.tight_layout()
plt.savefig('figures/GM_WM_dotplot.tiff')
plt.savefig('figures/GM_WM_dotplot.svg', format = 'svg')


In [ ]:
# plot regions verts

i = 0
j = 0
fig,ax=plt.subplots(4,2,figsize=(10,16))

for im in adataScaled.obs.ImageID.unique():
    ss = adataScaled[(adataScaled.obs.ImageID.isin([im]))]#sc.pp.subsample(adataScaled[adataScaled.obs.Region!=10],fraction=0.8, copy=True)
    cDict = {'GM':'tab:blue','WM':'tab:pink', 'not_annotated':'gray'}
    ss.uns['Region_colors']=[cDict[x] for x in sorted(ss.obs.Region.unique())]
    print(im, [x for x in sorted(ss.obs.Region.unique())],len(ss.uns['Region_colors']))

    if ss.obs.ImageType.values[0] == 'Ctl':
        if i<4:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[i][0],size=12.0,title=im,alpha=0.9, 
                      show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[i][0],size=12.0,title=im,alpha=0.9, 
                      show=False, color_map=cMapDict)
        i+=1
    else:
        if j<1:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[j][1],size=12.0,title=im,alpha=0.9,
                          show=False)
        else:
            sc.pl.scatter(ss, basis='spatial',color=['Region'],ax=ax[j][1],size=12.0,title=im,alpha=0.9, show=False,legend_loc='none')
        j+=1
plt.tight_layout()
plt.savefig('figures/GM_WM_dotplot_vertical.tiff')
plt.savefig('figures/GM_WM_dotplot_vertical.svg', format = 'svg')


In [ ]:
# def plotPie(adt, ax, axis_id):
#     count_DF  = adt.obs[adataScaled2.obs.Parent=='Grey Matter'].sample(frac=1).leiden.value_counts()
#     keepOrder = count_DF.index
#     labels = []
#     for ix, x in count_DF.items():
#         if (100.*x/count_DF.sum())>=1:
#             labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
#         else:
#             if  "< 1%" not in labels:
#                 labels.append("< 1%")
#             else:
#                 labels.append("")
#     ax[0][axis_id].pie(count_DF,explode=[0.05]*len(adt.obs.leiden.unique()),pctdistance=-0.85,startangle=0,
#             labels=labels, colors=[cMapDict[c] for c in keepOrder],
#           textprops = {"fontsize":12})
#     #plt.set_cmap('Set2')
#     #draw circle
#     centre_circle = plt.Circle((0,0),0.50,fc='white')
#     #fig = plt.gcf()
#     ax[0][axis_id].add_artist(centre_circle)
#     plt.tight_layout()
#     count_DF  = adt.obs[adt.obs.Parent=='White Matter'].sample(frac=1).leiden.value_counts()
#     keepOrder = count_DF.index
#     labels = []
#     for ix, x in count_DF.items():
#         if (100.*x/count_DF.sum())>=1:
#             labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
#         else:
#             if  "< 1%" not in labels:
#                 labels.append("< 1%")
#             else:
#                 labels.append("")
#     ax[1][axis_id].pie(count_DF.loc[keepOrder],explode=[0.05]*len(adt.obs.leiden.unique()),pctdistance=-0.85,startangle=0,
#             labels=labels, colors=[cMapDict[c] for c in keepOrder],
#           textprops = {"fontsize":12})
#     #draw circle
#     centre_circle = plt.Circle((0,0),0.50,fc='white')
#     #fig = plt.gcf()
#     ax[1][axis_id].add_artist(centre_circle)
    
    
    
    
    
def plotPie_custom(adt, ax, axis_id, cmap, feature):
    count_DF  = adt.obs[adataScaled2.obs.Parent=='Grey Matter'].sample(frac=1)[feature].value_counts()
    keepOrder = count_DF.index
    labels = []
    for ix, x in count_DF.items():
        print(100.*x/count_DF.sum())
        if (100.*x/count_DF.sum())>=1:
            labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
        else:
            if  "< 1%" not in labels:
                labels.append("< 1%")
            else:
                labels.append("")
    ax[0][axis_id].pie(count_DF,explode=None,pctdistance=-0.85,startangle=0,
            labels=labels, colors=[cmap[c] for c in keepOrder],
          textprops = {"fontsize":12})
    #plt.set_cmap('Set2')
    #draw circle
    centre_circle = plt.Circle((0,0),0.50,fc='white')
    #fig = plt.gcf()
    ax[0][axis_id].add_artist(centre_circle)
    plt.tight_layout()
    count_DF  = adt.obs[adt.obs.Parent=='White Matter'].sample(frac=1)[feature].value_counts()
    keepOrder = count_DF.index
    labels = []
    for ix, x in count_DF.items():
        if (100.*x/count_DF.sum())>=1:
            labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
        else:
            if  "< 1%" not in labels:
                labels.append("< 1%")
            else:
                labels.append("")
    ax[1][axis_id].pie(count_DF.loc[keepOrder],explode=None,pctdistance=-0.85,startangle=0,
            labels=labels, colors=[cmap[c] for c in keepOrder],
          textprops = {"fontsize":12})
    #draw circle
    centre_circle = plt.Circle((0,0),0.50,fc='white')
    #fig = plt.gcf()
    ax[1][axis_id].add_artist(centre_circle)
    print('')
    
    
    
    
    
    


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(2,3,figsize=(16,8))
adataScaled2 = adataScaled[adataScaled.obs.ImageType=='AD'].copy()
plotPie_custom(adataScaled2, ax, 0, cMapDict, 'leiden')
ax[0][0].set_title('AD - Grey Matter')
ax[1][0].set_title('AD - White Matter')
plt.tight_layout()

# exclude Ctl-2 sample, which corresponds to a patient with AML
adataScaled2 = adataScaled[adataScaled.obs.ImageID.isin(['Ctl-1','Ctl-3','Ctl-4'])].copy()
plotPie_custom(adataScaled2, ax, 1, cMapDict, 'leiden')
ax[0][1].set_title('Ctl - Grey Matter, AML excluded')
ax[1][1].set_title('Ctl - White Matter, AML excluded')

# plot for all control samples
adataScaled2 = adataScaled[adataScaled.obs.ImageType=='Ctl'].copy()
plotPie_custom(adataScaled2, ax, 2, cMapDict, 'leiden')
ax[0][2].set_title('Ctl - Grey Matter, AML included')
ax[1][2].set_title('Ctl - White Matter, AML included')
plt.tight_layout()
plt.savefig('figures/pieplots.tiff')
plt.savefig('figures/pieplots.svg', format = 'svg')



In [ ]:

import matplotlib.pyplot as plt
fig,ax=plt.subplots(2,4,figsize=(16,8))
adataScaled2 = adataScaled[adataScaled.obs.ImageType=='AD'].copy()
plotPie_custom(adataScaled2, ax, 0, cMapDict, 'leiden')
ax[0][0].set_title('AD - Grey Matter')
ax[1][0].set_title('AD - White Matter')
plt.tight_layout()

# exclude Ctl-2 sample, which corresponds to a patient with AML
adataScaled2 = adataScaled[adataScaled.obs.ImageID.isin(['Ctl-1','Ctl-3','Ctl-4'])].copy()
plotPie_custom(adataScaled2, ax, 1, cMapDict, 'leiden')
ax[0][1].set_title('Ctl - Grey Matter, AML excluded')
ax[1][1].set_title('Ctl - White Matter, AML excluded')

# plot for all control samples
adataScaled2 = adataScaled[adataScaled.obs.ImageType=='Ctl'].copy()
plotPie_custom(adataScaled2, ax, 2, cMapDict, 'leiden')
ax[0][2].set_title('Ctl - Grey Matter, AML included')
ax[1][2].set_title('Ctl - White Matter, AML included')

# plot for all control samples
adataScaled2 = adataScaled[adataScaled.obs.ImageID=='Ctl-2'].copy()
plotPie_custom(adataScaled2, ax, 3, cMapDict, 'leiden')
ax[0][3].set_title('Ctl - Grey Matter, AML only')
ax[1][3].set_title('Ctl - White Matter, AML only')

plt.tight_layout()
plt.savefig('figures/pieplots_AML.tiff')
plt.savefig('figures/pieplots_AML.svg', format = 'svg')






In [ ]:
# stats

# compare abundances between GM and WM

import copy

ad_control_dict = {'2997': 'AD',
 '3155': 'AD',
 '3196': 'AD',
 '3280': 'AD',
 '1796': 'Ctl',
 '3628': 'Ctl',
 '3026': 'Ctl', 
 '1873': 'Ctl'}



diseases = []
regions = []
samples_list = []

prop_Microglia_2 = []
prop_Microglia_1 = []
prop_BLM = []
prop_PVM = []
prop_Monocytes = []

counts_Microglia_2 = []
counts_Microglia_1 = []
counts_BLM = []
counts_PVM = []
counts_Monocytes = []




samples = adataScaled.obs.ImageIDOLD.unique()

for sample in samples:
    
    sample_disease = ad_control_dict[str(sample)]
    
    sample_sub_GM = adataScaled[(adataScaled.obs.ImageIDOLD == sample) & (adataScaled.obs.Parent == 'Grey Matter')]
    counts_GM = sample_sub_GM.obs.leiden.value_counts()
    counts_GM_normalized = counts_GM/counts_GM.sum()
    
    diseases.append(sample_disease)
    regions.append('GM')
    samples_list.append(sample)

    
    prop_Microglia_2.append(counts_GM_normalized['Microglia 2'])
    prop_Microglia_1.append(counts_GM_normalized['Microglia 1'])
    prop_BLM.append(counts_GM_normalized['BLM'])
    prop_PVM.append(counts_GM_normalized['PVM'])
    prop_Monocytes.append(counts_GM_normalized['Monocytes'])
    
    
    
    counts_Microglia_2.append(counts_GM['Microglia 2'])
    counts_Microglia_1.append(counts_GM['Microglia 1'])
    counts_BLM.append(counts_GM['BLM'])
    counts_PVM.append(counts_GM['PVM'])
    counts_Monocytes.append(counts_GM['Monocytes'])
    
    
    
    sample_sub_WM = adataScaled[(adataScaled.obs.ImageIDOLD == sample) & (adataScaled.obs.Parent == 'White Matter')]
    counts_WM = sample_sub_WM.obs.leiden.value_counts()
    counts_WM_normalized = counts_WM/counts_WM.sum()
    
    diseases.append(sample_disease)
    regions.append('WM')
    samples_list.append(sample)

    
    prop_Microglia_2.append(counts_WM_normalized['Microglia 2'])
    prop_Microglia_1.append(counts_WM_normalized['Microglia 1'])
    prop_BLM.append(counts_WM_normalized['BLM'])
    prop_PVM.append(counts_WM_normalized['PVM'])
    prop_Monocytes.append(counts_WM_normalized['Monocytes'])
    
    
    
    counts_Microglia_2.append(counts_WM['Microglia 2'])
    counts_Microglia_1.append(counts_WM['Microglia 1'])
    counts_BLM.append(counts_WM['BLM'])
    counts_PVM.append(counts_WM['PVM'])
    counts_Monocytes.append(counts_WM['Monocytes'])
    
    
    
prop_df = pd.DataFrame({'disease': diseases, 'region': regions, 'Microglia 2':prop_Microglia_2,'Microglia 1':prop_Microglia_1, 'BLM':prop_BLM, 'PVM':prop_PVM, 'Monocytes':prop_Monocytes, 'sample':samples_list})
counts_df = pd.DataFrame({'disease': diseases, 'region': regions, 'Microglia 2':counts_Microglia_2,'Microglia 1':counts_Microglia_1, 'BLM':counts_BLM, 'PVM':counts_PVM, 'Monocytes':counts_Monocytes, 'sample':samples_list})

area_file_path = 'sample_area_files/'

areas = []
for row in counts_df.index:
    rowvals = counts_df.loc[row]
    sam = rowvals['sample']
    reg = rowvals['region']
    
    area_table = pd.read_csv(area_file_path + sam + '.csv')
    
    if reg == 'GM':
        ar = area_table[area_table['zone'] == 'Grey Matter']['area_mm'].iloc[0]
    elif reg == 'WM':
        ar = area_table[area_table['zone'] == 'White Matter']['area_mm'].iloc[0]
    areas.append(ar)
    


cells_per_mm2_df = copy.deepcopy(counts_df)
cells_per_mm2_df['areas'] = areas

for celltype in set(adataScaled.obs.leiden):
    cells_per_mm2_df[celltype] = cells_per_mm2_df[celltype]/cells_per_mm2_df['areas']

# count_DF  = adataScaled.obs[adataScaled.obs.Parent=='Grey Matter'].sample(frac=1).leiden.value_counts()
# count_DF/count_DF.sum()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt





def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95, savepath = False):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(7, 5))
    plt.suptitle(title, fontsize=10)  # Title for the whole plot

    # Bar plot
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('Proportion')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    # plt.subplot(1, 2, 2)
    # plt.boxplot([data1, data2], labels=[data1label, data2label], )
    # plt.ylabel('Proportion')
    # plt.title('Box and Whisker Plot')

    # Annotations
    
    logfc = np.log2(mean1/mean2)
    

    loc = np.mean([np.max(data1.append(data2)), 0])
    adj = 0.1 * (np.max(data1.append(data2)))
    plt.text(3, loc + adj, annotation1, fontsize=10, ha='center')
    plt.text(3, loc - adj, 'log2fc: ' + str(float("{:.3e}".format(logfc))), fontsize=10, ha='center')

    plt.tight_layout()
    
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()







# some t tests, between ad and control of GM

from scipy.stats import ttest_ind

cell_types = adataScaled.obs.leiden.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = prop_df[prop_df.region == 'GM']
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    print(AD_props)
    print(Ctl_props)
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM

print('')
print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = prop_df[prop_df.region == 'WM']
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
    
    
    
    
    
# some t tests, between ad and control of GM AML removed

from scipy.stats import ttest_ind

cell_types = adataScaled.obs.leiden.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = prop_df[(prop_df.region == 'GM') & (prop_df['sample'] != '3628')]
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    print(AD_props)
    print(Ctl_props)
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM AML removed

print('')
print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = prop_df[(prop_df.region == 'WM') & (prop_df['sample'] != '3628')]
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
    
    
    
    
    
    
    
    
    

    
    
    
    
    
    
    
    
    
# within AD, test between GM and WM

print('')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = prop_df[prop_df.disease == 'AD']
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = prop_df[prop_df.disease == 'Ctl']
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of Ctl ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
# within AD, test between GM and WM AML removed

print('')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = prop_df[(prop_df.disease == 'AD') & (prop_df['sample'] != '3628')]
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM AML removed

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = prop_df[(prop_df.disease == 'Ctl') & (prop_df['sample'] != '3628')]
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of Ctl AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
# both ad and ctl, test between GM and WM

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    all_samps = copy.deepcopy(prop_df)
    
    
    GM_props = all_samps[all_samps.region == 'GM'][cell_type]    
    WM_props = all_samps[all_samps.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of all samples ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
# both ad and ctl, test between GM and WM AML exlcuded

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    all_samps = prop_df[(prop_df['sample'] != '3628')]
    
    GM_props = all_samps[all_samps.region == 'GM'][cell_type]    
    WM_props = all_samps[all_samps.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of all samples AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95, savepath = False):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.suptitle(title, fontsize=16)  # Title for the whole plot

    # Bar plot
    plt.subplot(1, 2, 1)
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('cells per mm2')
    plt.title('Comparison of Data with Confidence Intervals')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    plt.subplot(1, 2, 2)
    plt.boxplot([data1, data2], labels=[data1label, data2label], )
    plt.ylabel('cells per mm2')
    plt.title('Box and Whisker Plot')

    # Annotations
    

    loc = np.mean([np.max(data1.append(data2)), np.min(data1.append(data2))])
    adj = 0.1 * (np.max(data1.append(data2)) - np.min(data1.append(data2)))
    plt.text(3, loc + adj, annotation1, fontsize=10, ha='center')
    plt.text(3, loc - adj, annotation2, fontsize=10, ha='center')

    plt.tight_layout()
    
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()


def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95, savepath = False):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(7, 5))
    plt.suptitle(title, fontsize=10)  # Title for the whole plot

    # Bar plot
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('cells per mm2')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    # plt.subplot(1, 2, 2)
    # plt.boxplot([data1, data2], labels=[data1label, data2label], )
    # plt.ylabel('Proportion')
    # plt.title('Box and Whisker Plot')

    # Annotations
    
    logfc = np.log2(mean1/mean2)
    

    loc = np.mean([np.max(data1.append(data2)), 0])
    adj = 0.1 * (np.max(data1.append(data2)))
    plt.text(3, loc + adj, annotation1, fontsize=10, ha='center')
    plt.text(3, loc - adj, 'log2fc: ' + str(float("{:.3e}".format(logfc))), fontsize=10, ha='center')

    plt.tight_layout()
    
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()













# some t tests, between ad and control of GM

from scipy.stats import ttest_ind

cell_types = adataScaled.obs.leiden.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = cells_per_mm2_df[cells_per_mm2_df.region == 'GM']
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    print(AD_props)
    print(Ctl_props)
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM

print('')
print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = cells_per_mm2_df[cells_per_mm2_df.region == 'WM']
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
    
    
    
    
    
# some t tests, between ad and control of GM AML removed

from scipy.stats import ttest_ind

cell_types = adataScaled.obs.leiden.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = cells_per_mm2_df[(cells_per_mm2_df.region == 'GM') & (cells_per_mm2_df['sample'] != '3628')]
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    print(AD_props)
    print(Ctl_props)
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM AML removed

print('')
print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = cells_per_mm2_df[(cells_per_mm2_df.region == 'WM') & (cells_per_mm2_df['sample'] != '3628')]
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
    
    
    
    
    
    
    
    
    

    
    
    
    
    
    
    
    
    
# within AD, test between GM and WM

print('')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = cells_per_mm2_df[cells_per_mm2_df.disease == 'AD']
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = cells_per_mm2_df[cells_per_mm2_df.disease == 'Ctl']
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of Ctl ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
# within AD, test between GM and WM AML removed

print('')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = cells_per_mm2_df[(cells_per_mm2_df.disease == 'AD') & (cells_per_mm2_df['sample'] != '3628')]
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM AML removed

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = cells_per_mm2_df[(cells_per_mm2_df.disease == 'Ctl') & (cells_per_mm2_df['sample'] != '3628')]
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of Ctl AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')

In [ ]:
from scipy.stats import chi2_contingency
import copy


# chi squared

# is there a significant relationship between AD and cell type proportions within GM?



def calculate_standardized_residuals(observed, expected):
    row_totals = np.sum(observed, axis=1)
    column_totals = np.sum(observed, axis=0)
    grand_total = np.sum(observed)
    
    standardized_residuals = (observed - expected) / np.sqrt(expected * (1 - row_totals[:, np.newaxis] / grand_total) * (1 - column_totals / grand_total))
    
    return standardized_residuals







print('GM')

prop_df_hundred = prop_df[cell_types]*100
prop_df_hundred['disease'] = prop_df['disease']
prop_df_hundred['region'] = prop_df['region']

#counts_df = prop_df_hundred

    
GM_sub = counts_df[counts_df.region == 'GM']

AD_props = GM_sub[GM_sub.disease == 'AD'][cell_types].mean()
Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_types].mean()

observed = [AD_props, Ctl_props]

chi2, p, dof, expected = chi2_contingency(observed)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:")
print(expected)

residuals = calculate_standardized_residuals(observed, expected)
significant_deviations = np.where(np.abs(residuals) > 2)

resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['AD'] * 5) + (['Ctl'] * 5)})

print(resid_df)

sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
plt.title('GM ad vs control')
plt.xlabel('Cell Types')
plt.ylabel('Residuals')
plt.legend(loc='upper left')
plt.show()




print('WM')


    
WM_sub = counts_df[counts_df.region == 'WM']

AD_props = WM_sub[WM_sub.disease == 'AD'][cell_types].mean()
Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_types].mean()

observed = [AD_props, Ctl_props]

chi2, p, dof, expected = chi2_contingency(observed)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:")
print(expected)

residuals = calculate_standardized_residuals(observed, expected)
significant_deviations = np.where(np.abs(residuals) > 2)

resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['AD'] * 5) + (['Ctl'] * 5)})
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
plt.title('WM ad vs control')
plt.xlabel('Cell Types')
plt.ylabel('Residuals')
plt.legend(loc='upper left')
plt.show()





print('AD')


    
AD_sub = counts_df[counts_df.disease == 'AD']

GM_props = AD_sub[AD_sub.region == 'GM'][cell_types].mean()
WM_props = AD_sub[AD_sub.region == 'WM'][cell_types].mean()

observed = [GM_props, WM_props]

chi2, p, dof, expected = chi2_contingency(observed)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:")
print(expected)

residuals = calculate_standardized_residuals(observed, expected)
significant_deviations = np.where(np.abs(residuals) > 2)

resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['GM'] * 5) + (['WM'] * 5)})
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
plt.title('AD gm vs wm')
plt.xlabel('Cell Types')
plt.ylabel('Residuals')
plt.legend(loc='upper left')
plt.show()



print('Ctl')


    
Ctl_sub = counts_df[counts_df.disease == 'Ctl']

GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_types].mean()
WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_types].mean()

observed = [GM_props, WM_props]

chi2, p, dof, expected = chi2_contingency(observed)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:")
print(expected)

residuals = calculate_standardized_residuals(observed, expected)
significant_deviations = np.where(np.abs(residuals) > 2)

resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['GM'] * 5) + (['WM'] * 5)})
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
plt.title('Ctl gm vs wm')
plt.xlabel('Cell Types')
plt.ylabel('Residuals')
plt.legend(loc='upper left')
plt.show()



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Creating example DataFrame


# Set the style of the seaborn plot
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
plt.title('Values vs Cell Types (Colored by Category)')
plt.xlabel('Cell Types')
plt.ylabel('Residuals')
plt.legend(loc='upper left')
plt.show()



In [ ]:
## Plot AD statistics
# dt = 'Distance to nearest Plaque (µm)'
# ss = adataScaled[(adataScaled.obs.ImageType =='AD')]
# ss = ss[ss.obs.Parent=='Grey Matter']
# BINS  =[-1,0,5,10,15,20,30,50,75,100,200,500]
# out = pd.cut(ss.obs[dt],bins=BINS, right=True)
# ss.obs['Quantile'] = out.values
 

In [ ]:


# fig,ax = plt.subplots(2,1,figsize=(8,8),gridspec_kw={'height_ratios': [1.6,1.2]})

# matpl=sc.pl.matrixplot(ss, var_names = adataScaled.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
# matpl.legend(show=False)


# import numpy as np
# sns.set_style("whitegrid", {'axes.grid' : False})




# oDF = ss.obs.groupby(['Quantile','leiden','ImageID']).size().T.reset_index().groupby(['Quantile','leiden']).mean().unstack(level=1).T.reset_index(level = 0,drop=True)
# oDFPct = 100*oDF/oDF.sum(axis=0)

# oDFPct.T.plot(kind='bar',stacked=True,ax=ax[0],color=cMapDict, legend=True, )
# ax[0].legend(title='Cluster ID',loc=(1,0.0))
# ax[0].set_ylabel('Percentage of \n Microglial cells')
# ax[0].set_xlabel('')
# ax[0].set_xticks([])



# # find MG dections that are clusters
# oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

# oDFPct = oDF.T#/oDF.sum(axis=1)
# oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
# ax[1].set_ylabel('% of cells that \n are Microglial clusters')
# ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
# ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
# ax[1].set_ylim(0,40)

# plt.tight_layout()
# plt.savefig('pixel_clustering_newmethod/paper_images/distance_analysis.tiff')



### morph

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial") 

sns.set_style('white')

change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
}

new_name_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']
colors_in_order = [cMapDict[i] for i in new_name_order]
cMapDict = dict(zip(new_name_order, colors_in_order))





change_cluster_names_dict_morph = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

new_name_order_morph = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
colors_in_order_morph = [cMapDict_morph[i] for i in new_name_order_morph]
cMapDict_morph = dict(zip(new_name_order_morph, colors_in_order_morph))


change_cluster_names_dict_morph_presub = {
          1: 'Rounded',
          2: 'Intermediate',
          0: 'Ramified',                                       
}


cMapDict_morph_presub = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified':'goldenrod'
}

new_name_order_morph_presub = ['Rounded', 'Intermediate', 'Ramified']
colors_in_order_morph_presub = [cMapDict_morph_presub[i] for i in new_name_order_morph_presub]
cMapDict_morph_presub = dict(zip(new_name_order_morph_presub, colors_in_order_morph_presub))


# adataScaled.obs['leiden'] = [change_cluster_names_dict[i] for i in adataScaled.obs['leiden']]
# adataScaled.obs['leiden'] = pd.Categorical(adataScaled.obs['leiden'], categories=new_name_order, ordered=False)






morph_data_orig = sc.read_h5ad('final_h5ads/clustering_and_morph_clustering.h5')

morph_data_orig = morph_data_orig[morph_data_orig.obs['Parent'].isin(['Grey Matter', 'White Matter'])]

morph_data_orig.obs['protein_leiden_updated'] = [change_cluster_names_dict[i] for i in morph_data_orig.obs.protein_leiden]
morph_data_orig.obs['protein_leiden_updated'] = pd.Categorical(morph_data_orig.obs['protein_leiden_updated'], categories=new_name_order, ordered=False)


morph_data_orig.obs['final_leiden_M1_updated'] = [change_cluster_names_dict_morph[i] for i in morph_data_orig.obs.final_leiden_M1]
morph_data_orig.obs['final_leiden_M1_updated'] = pd.Categorical(morph_data_orig.obs['final_leiden_M1_updated'], categories=new_name_order_morph, ordered=False)


morph_data_orig.obs['leidenM1_orig_updated'] = [change_cluster_names_dict_morph_presub[i] for i in morph_data_orig.obs.leidenM1_orig]
morph_data_orig.obs['leidenM1_orig_updated'] = pd.Categorical(morph_data_orig.obs['leidenM1_orig_updated'], categories=new_name_order_morph_presub, ordered=False)



protein_data_matrix = morph_data_orig.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_data_matrix = morph_data_orig.to_df()

# new_obs = pd.merge(adataScaled.obs, morph_data_orig.obs['leidenM1'], left_index=True, right_index=True, how='left')

# adataScaled.obs = new_obs
# morph_data = adataScaled
# morph_data.obs.index = morph_data.obs['ImageID_CellID']
# morph_data = morph_data[[not i for i in morph_data.obs['leidenM1'].isna()]]

# morph_data.obs['leidenM1'] = pd.Categorical(morph_data.obs['leidenM1'])

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


# Define custom color palette scaling from white to blue
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

sc.pl.tsne(morph_data_orig, color=morph_data_orig.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False, layer='unscaled_morph_data')
plt.savefig('figures/morph_figures/marker_feature_plot_bluewhite_noscaling.tiff')
plt.savefig('figures/morph_figures/marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')

In [ ]:


import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import h5py


img_dict = {
    '1796': '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.h5',
    '3196': '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.h5',
    '2997': '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.h5',
    '1873': '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.h5',
    '3026': '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.h5',
    '3155': '11162021_Bahareh-3155_EDTA_IBA1_cell1.h5',
    '3280': '12032021-Bahareh_3280_Alz_IBA1_cell1.h5',
    '3628': '12252021_Bahareh-3628-Cntr_IBA1_cell1.h5'
}

def getInputs(imID):
    #print(f"Reading image file: Bahareh/GMask/{imID}_labeledMG.tif" )
    #inImg = tf.TiffFile(f'Bahareh/GMask/{imID}_labeledMG.tif').pages[0].asarray()
    
    impath = 'segmentation_files/'
    #impath = 'segmentation_files/'
    imname = img_dict[imID]
    
    with h5py.File(impath + imname, 'r') as file:
        inImg = np.array(file['matrix'])
    
    return inImg


import tifffile as tf
def getCrop(inM, keepID):
    # Find the coordinates of all non-zero pixels with the specified keepID
    non_zero_coords = np.argwhere(inM == keepID)
    
    
    x_coords = [point[0] for point in non_zero_coords]
    y_coords = [point[1] for point in non_zero_coords]
    
    center_x = sum(x_coords) / len(x_coords)
    center_y = sum(y_coords) / len(y_coords)
    
    # Extract the minimum and maximum x and y coordinates
    x_min, y_min = np.min(non_zero_coords, axis=0)
    x_max, y_max = np.max(non_zero_coords, axis=0)
    # Calculate width (w) and height (h) based on the min and max coordinates
    # pad by 50
    pad = 50
    #w = x_max - x_min + pad
    #h = y_max - y_min + pad
    
    
    
    
    return (inM[max(0,x_min-pad):min(x_max+pad, inM.shape[0]), max(y_min-pad,0):min(y_max+pad, inM.shape[1])]==keepID).astype(np.uint8)




def getCrop_fixed(inM, keepID):
    # Find the coordinates of all non-zero pixels with the specified keepID
    non_zero_coords = np.argwhere(inM == keepID)
    height, width = inM.shape
    
    x_coords = [point[0] for point in non_zero_coords]
    y_coords = [point[1] for point in non_zero_coords]
    
    center_x = int(sum(x_coords) / len(x_coords))
    center_y = int(sum(y_coords) / len(y_coords))
    
    
    
    
    
    
    buffer_size_hori = 60
    buffer_size_vert = 60
    
    start_x = max(0, center_x - buffer_size_hori)
    end_x = min(height, center_x + buffer_size_hori)
    start_y = max(0, center_y - buffer_size_vert)
    end_y = min(width, center_y + buffer_size_vert)

    
    # Calculate the cropped image
    
    print(start_x)
    print(end_x)
    print(start_y)
    print(end_y)
    
    
    #return (inM[max(0,x_min-pad):min(x_max+pad, inM.shape[0]), max(y_min-pad,0):min(y_max+pad, inM.shape[1])]==keepID).astype(np.uint8)

    cropped_image = (inM[start_x:end_x, start_y:end_y]==keepID).astype(np.uint8)
    
    return(cropped_image)




def getSampleImages(adata, feature):
    # calculate the average expression of each gene in each cluster
    cluster_means = {}
    #for clusterID in np.unique(adata.obs[feature]):
        
    for clusterID in list(adata.obs[feature].unique().categories):
        
        
        cluster_cells = adata[adata.obs[feature] == clusterID].X.toarray()
        cluster_mean = np.mean(cluster_cells, axis=0)
        cluster_means[clusterID] = cluster_mean
        
    # sample four cells that are most representative of each cluster
    fig, ax = plt.subplots(len(np.unique(adata.obs[feature])),4,figsize=(16, 4 * len(np.unique(adata.obs[feature]))))
    for ia, (clusterID, cluster_mean) in enumerate(cluster_means.items()):
        print(f"Cluster ID: {clusterID}")
        cellIDs = adata[adata.obs[feature] == clusterID].X.toarray()
        similarities = cosine_similarity(cellIDs, cluster_mean.reshape(1, -1))
        cellIDs = adata.obs[(adata.obs[feature]==clusterID)].iloc[np.argsort(similarities.ravel())[-4:]].ImageID_CellID.values[::-1]
        print(cellIDs)
        for ix,cID in enumerate(cellIDs):
            print(cID.split('_')[0])
            print(cID.split('_')[1])
            inIm = getInputs(cID.split('_')[0])
            inIm = getCrop_fixed(inIm, int(cID.split('_')[1]))
            
            ax[ia][ix].imshow(inIm)
            ax[ia][ix].xaxis.grid(visible = False)
            ax[ia][ix].yaxis.grid(visible = False)
            ax[ia][ix].set_title(f"{clusterID}_{cID}")
    
    
    return(fig)     

    
sample_images_fig = getSampleImages(morph_data_orig, 'final_leiden_M1_updated')
sample_images_fig.savefig('figures/morph_figures/sample_images.tiff', format = 'tiff')
sample_images_fig.savefig('figures/morph_figures/sample_images.svg', format = 'svg')


sample_images_fig_presub = getSampleImages(morph_data_orig, 'leidenM1_orig_updated')
sample_images_fig.savefig('figures/morph_figures/sample_images_presub.tiff', format = 'tiff')
sample_images_fig.savefig('figures/morph_figures/sample_images_presub.svg', format = 'svg')


In [ ]:


import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import h5py


img_dict = {
    '1796': '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.h5',
    '3196': '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.h5',
    '2997': '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.h5',
    '1873': '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.h5',
    '3026': '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.h5',
    '3155': '11162021_Bahareh-3155_EDTA_IBA1_cell1.h5',
    '3280': '12032021-Bahareh_3280_Alz_IBA1_cell1.h5',
    '3628': '12252021_Bahareh-3628-Cntr_IBA1_cell1.h5'
}

def getInputs(imID):
    #print(f"Reading image file: Bahareh/GMask/{imID}_labeledMG.tif" )
    #inImg = tf.TiffFile(f'Bahareh/GMask/{imID}_labeledMG.tif').pages[0].asarray()
    
    impath = 'segmentation_files/'
    imname = img_dict[imID]
    
    with h5py.File(impath + imname, 'r') as file:
        inImg = np.array(file['matrix'])
    
    return inImg


import tifffile as tf
def getCrop(inM, keepID):
    # Find the coordinates of all non-zero pixels with the specified keepID
    non_zero_coords = np.argwhere(inM == keepID)
    
    
    x_coords = [point[0] for point in non_zero_coords]
    y_coords = [point[1] for point in non_zero_coords]
    
    center_x = sum(x_coords) / len(x_coords)
    center_y = sum(y_coords) / len(y_coords)
    
    # Extract the minimum and maximum x and y coordinates
    x_min, y_min = np.min(non_zero_coords, axis=0)
    x_max, y_max = np.max(non_zero_coords, axis=0)
    # Calculate width (w) and height (h) based on the min and max coordinates
    # pad by 50
    pad = 50
    #w = x_max - x_min + pad
    #h = y_max - y_min + pad
    
    
    
    
    return (inM[max(0,x_min-pad):min(x_max+pad, inM.shape[0]), max(y_min-pad,0):min(y_max+pad, inM.shape[1])]==keepID).astype(np.uint8)




def getCrop_fixed(inM, keepID):
    # Find the coordinates of all non-zero pixels with the specified keepID
    non_zero_coords = np.argwhere(inM == keepID)
    height, width = inM.shape
    
    x_coords = [point[0] for point in non_zero_coords]
    y_coords = [point[1] for point in non_zero_coords]
    
    center_x = int(sum(x_coords) / len(x_coords))
    center_y = int(sum(y_coords) / len(y_coords))
    
    
    
    
    
    
    buffer_size_hori = 60
    buffer_size_vert = 60
    
    start_x = max(0, center_x - buffer_size_hori)
    end_x = min(height, center_x + buffer_size_hori)
    start_y = max(0, center_y - buffer_size_vert)
    end_y = min(width, center_y + buffer_size_vert)

    
    # Calculate the cropped image
    
    print(start_x)
    print(end_x)
    print(start_y)
    print(end_y)
    
    
    #return (inM[max(0,x_min-pad):min(x_max+pad, inM.shape[0]), max(y_min-pad,0):min(y_max+pad, inM.shape[1])]==keepID).astype(np.uint8)

    cropped_image = (inM[start_x:end_x, start_y:end_y]==keepID).astype(np.uint8)
    
    return(cropped_image)




def getSampleImages(adata, feature, num_cells = 4):
    # calculate the average expression of each gene in each cluster
    cluster_means = {}
    #for clusterID in np.unique(adata.obs[feature]):
        
    for clusterID in list(adata.obs[feature].unique().categories):
        
        
        cluster_cells = adata[adata.obs[feature] == clusterID].X.toarray()
        cluster_mean = np.mean(cluster_cells, axis=0)
        cluster_means[clusterID] = cluster_mean
        
    # sample four cells that are most representative of each cluster
    fig, ax = plt.subplots(len(np.unique(adata.obs[feature])),num_cells,figsize=(16, num_cells * len(np.unique(adata.obs[feature]))))
    for ia, (clusterID, cluster_mean) in enumerate(cluster_means.items()):
        print(f"Cluster ID: {clusterID}")
        cellIDs = adata[adata.obs[feature] == clusterID].X.toarray()
        similarities = cosine_similarity(cellIDs, cluster_mean.reshape(1, -1))
        cellIDs = adata.obs[(adata.obs[feature]==clusterID)].iloc[np.argsort(similarities.ravel())[-num_cells:]].ImageID_CellID.values[::-1]
        print(cellIDs)
        for ix,cID in enumerate(cellIDs):
            print(cID.split('_')[0])
            print(cID.split('_')[1])
            inIm = getInputs(cID.split('_')[0])
            inIm = getCrop_fixed(inIm, int(cID.split('_')[1]))
            
            ax[ia][ix].imshow(inIm)
            ax[ia][ix].xaxis.grid(visible = False)
            ax[ia][ix].yaxis.grid(visible = False)
            ax[ia][ix].set_title(f"{clusterID}_{cID}")
    
    
    return(fig)     

    
sample_images_fig = getSampleImages(morph_data_orig, 'final_leiden_M1_updated', 10)
sample_images_fig.savefig('figures/morph_figures/sample_images_additional.tiff', format = 'tiff')
sample_images_fig.savefig('figures/morph_figures/sample_images_additional.svg', format = 'svg')


sample_images_fig_presub = getSampleImages(morph_data_orig, 'leidenM1_orig_updated', 10)
sample_images_fig.savefig('figures/morph_figures/sample_images_presub_additional.tiff', format = 'tiff')
sample_images_fig.savefig('figures/morph_figures/sample_images_presub_additional.svg', format = 'svg')


In [ ]:
morph_data_orig.obsm['X_spatial'] = np.array([morph_data_orig.obs.spatial_X, morph_data_orig.obs.spatial_Y]).T

# plot spatila maps
i = 0
j = 0
fig,ax=plt.subplots(2,4,figsize=(16,10))

for im in morph_data_orig.obs.ImageID.unique():
    ss = morph_data_orig[(morph_data_orig.obs.ImageID.isin([im]))]#sc.pp.subsample(morph_data_orig[morph_data_orig.obs.final_leiden_M1_updated!=10],fraction=0.8, copy=True)
    
    ss.uns['final_leiden_M1_updated_colors']=[cMapDict_morph[x] for x in ss.obs.final_leiden_M1_updated.unique().categories]
    print(im, [x for x in sorted(ss.obs.final_leiden_M1_updated.unique())],len(ss.uns['final_leiden_M1_updated_colors']))
    
    if ss.obs.ImageType.values[0] == 'Ctl':
        if i<3:
            sc.pl.scatter(ss, basis='spatial',color=['final_leiden_M1_updated'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['final_leiden_M1_updated'],ax=ax[0][i],size=12.0,title=im,alpha=0.9, 
                      show=False, color_map=cMapDict_morph)
        i+=1
    else:
        if j<4:
            sc.pl.scatter(ss, basis='spatial',color=['final_leiden_M1_updated'],ax=ax[1][j],size=12.0,title=im,alpha=0.9,
                          show=False,legend_loc='none')
        else:
            sc.pl.scatter(ss, basis='spatial',color=['final_leiden_M1_updated'],ax=ax[1][j],size=12.0,title=im,alpha=0.9, show=False)
        j+=1
plt.tight_layout()
plt.savefig('figures/morph_figures/microglia_clusters_dotplot_MORPH.tiff')
plt.savefig('figures/morph_figures/microglia_clusters_dotplot_MORPH.svg', format = 'svg')



In [ ]:

genes_to_show = ['IBA1', 'TMEM119', 'CD74', 'CD14', 'CD163', 'CD68', 'CD11b', 'CD11c', 'HLA-DR', 'CD45']


# sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='final_leiden_M1_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)

custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)

custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_minmax.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_minmax.svg', layer = 'meta', var_names=genes_to_show)



# sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='final_leiden_M1_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression.svg', layer = 'unscaled_morph_data')

custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_Z.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_Z.svg', layer = 'unscaled_morph_data')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_minmax.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_minmax.svg', layer = 'unscaled_morph_data')






# sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='leidenM1_orig_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.svg', format = 'svg')


custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)


custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)

custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_minmax.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_minmax.svg', layer = 'meta', var_names=genes_to_show)


# sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='leidenM1_orig_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.svg', layer = 'unscaled_morph_data')

custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_Z.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_Z.svg', layer = 'unscaled_morph_data')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_minmax.tiff', layer='unscaled_morph_data')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_minmax.svg', layer = 'unscaled_morph_data')





In [ ]:

genes_to_show = ['IBA1', 'TMEM119', 'CD74', 'CD14', 'CD163', 'CD68', 'CD11b', 'CD11c', 'HLA-DR', 'CD45']




custom_scaled_heatmap_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)

custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)


custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_minmax.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_minmax.svg', layer = 'meta', var_names=genes_to_show)






# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.svg', format = 'svg')


custom_scaled_heatmap_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)


custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)


custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_minmax.tiff', layer='meta', var_names=genes_to_show)
custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_minmax.svg', layer = 'meta', var_names=genes_to_show)







In [ ]:

genes_to_show = ['IBA1', 'TMEM119', 'CD74', 'CD14', 'CD163', 'CD68', 'CD11b', 'CD11c', 'HLA-DR', 'CD45']


# sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='final_leiden_M1_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression_median.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression_median.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')

custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_minmax_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_minmax_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')



# sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='final_leiden_M1_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression_median.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression_median.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_median.svg', layer = 'unscaled_morph_data', style = 'median')

custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_Z_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_Z_median.svg', layer = 'unscaled_morph_data', style = 'median')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_minmax_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression_minmax_median.svg', layer = 'unscaled_morph_data', style = 'median')






# sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='leidenM1_orig_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_median.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_median.svg', format = 'svg')


custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')


custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_minmax_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_minmax_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')


# sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='leidenM1_orig_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_median.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_median.svg', format = 'svg')

custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_median.svg', layer = 'unscaled_morph_data', style = 'median')

custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_Z_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_Z_median.svg', layer = 'unscaled_morph_data', style = 'median')

custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_minmax_median.tiff', layer='unscaled_morph_data', style = 'median')
custom_scaled_heatmap_minmax(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression_minmax_median.svg', layer = 'unscaled_morph_data', style = 'median')





In [ ]:

genes_to_show = ['IBA1', 'TMEM119', 'CD74', 'CD14', 'CD163', 'CD68', 'CD11b', 'CD11c', 'HLA-DR', 'CD45']




custom_scaled_heatmap_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')

custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_Z_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_Z_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')


custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_minmax_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/dotmap_morph_clusters_protein_expression_minmax_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')






# fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_median.svg', format = 'svg')


custom_scaled_heatmap_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')


custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_Z_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_Z_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_Z_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')


custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_minmax_median.tiff', layer='meta', var_names=genes_to_show, style = 'median')
custom_scaled_heatmap_minmax_dot(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/dotmap_presub_morph_clusters_protein_expression_minmax_median.svg', layer = 'meta', var_names=genes_to_show, style = 'median')







In [ ]:
# genes_to_show = ['IBA1', 'TMEM119', 'CD74', 'CD14', 'CD163', 'CD68', 'CD11b', 'CD11c', 'HLA-DR', 'CD45']


# sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='final_leiden_M1_updated', 
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression.tiff', format = 'tiff')
# fig.savefig('figures/morph_figures/heatmap_morph_clusters_protein_expression.svg', format = 'svg')

# custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
# custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)

# custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
# custom_scaled_heatmap_Z(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)

# # sc.tl.dendrogram(morph_data_orig, groupby='final_leiden_M1_updated')
# # fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='final_leiden_M1_updated', 
# #                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# # fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression.tiff', format = 'tiff')
# # fig.savefig('figures/morph_figures/heatmap_morph_clusters_morph_expression.svg', format = 'svg')

# custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression.tiff', layer='unscaled_morph_data')
# custom_scaled_heatmap(morph_data_orig, group_by='final_leiden_M1_updated', saveto='figures/morph_figures/heatmap_morph_clusters_morph_expression.svg', layer = 'unscaled_morph_data')






# # sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# # fig = sc.pl.matrixplot(morph_data_orig, var_names = genes_to_show,groupby='leidenM1_orig_updated', 
# #                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# # fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.tiff', format = 'tiff')
# # fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.svg', format = 'svg')


# custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.tiff', layer='meta', var_names=genes_to_show)
# custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression.svg', layer = 'meta', var_names=genes_to_show)


# custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z.tiff', layer='meta', var_names=genes_to_show)
# custom_scaled_heatmap_Z(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_protein_expression_Z.svg', layer = 'meta', var_names=genes_to_show)


# # sc.tl.dendrogram(morph_data_orig, groupby='leidenM1_orig_updated')
# # fig = sc.pl.matrixplot(morph_data_orig, var_names = morph_data_orig.var_names,groupby='leidenM1_orig_updated', 
# #                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True,swap_axes=True)

# # fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.tiff', format = 'tiff')
# # fig.savefig('figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.svg', format = 'svg')

# custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.tiff', layer='unscaled_morph_data')
# custom_scaled_heatmap(morph_data_orig, group_by='leidenM1_orig_updated', saveto='figures/morph_figures/heatmap_presub_morph_clusters_morph_expression.svg', layer = 'unscaled_morph_data')



In [ ]:

# fig = plt.figure()
# sns.scatterplot(data=adataScaled.obs, x=adataScaled.obsm['X_tsne'][:,0], y=adataScaled.obsm['X_tsne'][:,1],
#                 hue='leiden', palette=cMapDict,s=1,)


# plt.legend(ncol=4,loc=(0,-0.5))
# plt.xlabel('tSNE1')
# plt.ylabel('tSNE2')
# plt.tight_layout()
# plt.savefig('figures/leiden_clustering_tsne.tiff')
# plt.savefig('figures/leiden_clustering_tsne.svg', format = 'svg')







In [ ]:



fig = plt.figure() 
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='final_leiden_M1_updated', palette=cMapDict_morph,s=1,)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/morph_clusters_tsne.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_clusters_tsne.svg', format = 'svg')


fig = plt.figure() 
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='leidenM1_orig_updated', palette=cMapDict_morph_presub,s=1,)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne.svg', format = 'svg')





fig = plt.figure() 
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='final_leiden_M1_updated', palette=cMapDict_morph,s=1,legend = False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/morph_clusters_tsne_nolabel.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_clusters_tsne_nolabel.svg', format = 'svg')


fig = plt.figure() 
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='leidenM1_orig_updated', palette=cMapDict_morph_presub,s=1,legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne_nolabel.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne_nolabel.svg', format = 'svg')






In [ ]:

sc.pl.scatter(morph_data_orig, basis = 'tsne', color='final_leiden_M1_updated', color_map=cMapDict_morph, palette=cMapDict_morph.values(), show=False, legend_loc='none', title='', size = 4)
plt.savefig('figures/morph_figures/morph_clusters_tsne_nolabel_EASYEDIT.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_clusters_tsne_nolabel_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(morph_data_orig, basis = 'tsne', color='leidenM1_orig_updated', color_map=cMapDict_morph_presub, palette=cMapDict_morph_presub.values(), show=False, legend_loc='none', title='', size = 4)
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne_nolabel_EASYEDIT.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/presub_morph_clusters_tsne_nolabel_EASYEDIT.svg', format = 'svg')






In [ ]:
fig = sc.pl.tsne(morph_data_orig, color=morph_data_orig.var_names,use_raw=False, return_fig = True)
fig.savefig('figures/morph_figures/morph_marker_tsne.tiff')
fig.savefig('figures/morph_figures/morph_marker_tsne.svg', format = 'svg')



In [ ]:


colors = list(cMapDict_morph.values())
leiden_clusters = list(set(morph_data_orig.obs['final_leiden_M1_updated']))



custom_key = lambda x: new_name_order_morph.index(x)

leiden_clusters = sorted(leiden_clusters, key=custom_key)


cMapDict_morph = {}

for i in zip(colors, leiden_clusters):
    print(i)
    cMapDict_morph[i[1]] = i[0]


regiondict = {'Grey Matter':'GM', 'White Matter':'WM', np.nan: 'not_annotated'}
morph_data_orig.obs['Region'] = [regiondict[i] for i in morph_data_orig.obs['Parent']]


In [ ]:

#sc.pl.tsne(morph_data_orig, color=['final_leiden_M1_updated','ImageTypev2','ImageType'], size=2,ncols=3,sort_order=False, save='_umapsV4.png')
#fig,ax=plt.subplots(1,4, figsize=(13,3.5))

# fig = plt.figure()
# sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
#                 hue='broad_final_leiden_M1_updated',s=1,)


# plt.legend(ncol=4,loc=(0,-0.4))
# plt.xlabel('tSNE1')
# plt.ylabel('tSNE2')
# plt.tight_layout()

# plt.savefig('pixel_clustering_newmethod/paper_images/broad_final_leiden_M1_updated_clustering_tsne.tiff')

# fig = plt.figure()



fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='final_leiden_M1_updated', palette=cMapDict_morph,s=1,)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/final_leiden_M1_updated_clustering_tsne.tiff')
plt.savefig('figures/morph_figures/final_leiden_M1_updated_clustering_tsne.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='leidenM1_orig_updated',s=1,palette=cMapDict_morph_presub)


plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/pre_subclustering_final_leiden_M1_updated_tsne_nodefault.tiff')
plt.savefig('figures/morph_figures/pre_subclustering_final_leiden_M1_updated_tsne_nodefault.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='Region',s=1, palette=['tab:blue','tab:pink'], hue_order=['GM','WM'])


plt.legend(ncol=4,loc=(0,-0.3))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/GM_WM_tsne.tiff')
plt.savefig('figures/morph_figures/GM_WM_tsne.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='ImageType',s=1,palette=sc.pl.palettes.godsnot_102[3:5], hue_order=['AD','Ctl'])

plt.legend(ncol=4,loc=(0,-0.3))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/AD_CNT_tsne.tiff')
plt.savefig('figures/morph_figures/AD_CNT_tsne.svg', format = 'svg')



fig = plt.figure()

sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='ImageID', palette=sc.pl.palettes.godsnot_102[9:17],s=1,  hue_order=sorted(list(morph_data_orig.obs['ImageID'].unique()))
)
plt.legend(ncol=4,loc=(0,-0.5))
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/sample_tsne.tiff')
plt.savefig('figures/morph_figures/sample_tsne.svg', format = 'svg')













fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='final_leiden_M1_updated', palette=cMapDict_morph,s=1,legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/final_leiden_M1_updated_clustering_tsne_nolegend.tiff')
plt.savefig('figures/morph_figures/final_leiden_M1_updated_clustering_tsne_nolegend.svg', format = 'svg')





fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='Region',s=1, palette=['tab:blue','tab:pink'], hue_order=['GM','WM'], legend=False)


plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/GM_WM_tsne_nolegend.tiff')
plt.savefig('figures/morph_figures/GM_WM_tsne_nolegend.svg', format = 'svg')


fig = plt.figure()
sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='ImageType',s=1,palette=sc.pl.palettes.godsnot_102[3:5], hue_order=['AD','Ctl'], legend=False)

plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/AD_CNT_tsne_nolegend.tiff')
plt.savefig('figures/morph_figures/AD_CNT_tsne_nolegend.svg')



fig = plt.figure()

sns.scatterplot(data=morph_data_orig.obs, x=morph_data_orig.obsm['X_tsne'][:,0], y=morph_data_orig.obsm['X_tsne'][:,1],
                hue='ImageID', palette=sc.pl.palettes.godsnot_102[9:17],s=1, legend=False, hue_order=sorted(list(morph_data_orig.obs['ImageID'].unique()))
)
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')
plt.tight_layout()
plt.savefig('figures/morph_figures/sample_tsne_nolegend.tiff')
plt.savefig('figures/morph_figures/sample_tsne_nolegend.svg')



















In [ ]:



sc.pl.scatter(morph_data_orig, basis = 'tsne', color='Region',palette=['tab:blue','tab:pink'], show=False, legend_loc='none', title='', size = 4)
plt.savefig('figures/morph_figures/GM_WM_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/morph_figures/GM_WM_tsne_nolegend_EASYEDIT.svg', format = 'svg')

sc.pl.scatter(morph_data_orig, basis = 'tsne', color='ImageType',palette=sc.pl.palettes.godsnot_102[3:5], show=False, legend_loc='none', title='', size = 4)
plt.savefig('figures/morph_figures/AD_CNT_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/morph_figures/AD_CNT_tsne_nolegend_EASYEDIT.svg')



sc.pl.scatter(morph_data_orig, basis = 'tsne', color='ImageID',palette=sc.pl.palettes.godsnot_102[9:17], show=False, legend_loc='none', title='', size = 4)
plt.savefig('figures/morph_figures/sample_tsne_nolegend_EASYEDIT.tiff')
plt.savefig('figures/morph_figures/sample_tsne_nolegend_EASYEDIT.svg')







In [ ]:
# def plotPie(adt, ax, axis_id, morph_cMapDict, clus_names):
#     count_DF  = adt.obs[adataScaled2.obs.Parent=='Grey Matter'].sample(frac=1)[clus_names].value_counts()
#     keepOrder = count_DF.index
#     labels = []
#     for ix, x in count_DF.items():
#         if (100.*x/count_DF.sum())>=1:
#             labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
#         else:
#             if  "< 1%" not in labels:
#                 labels.append("< 1%")
#             else:
#                 labels.append("")
#     ax[0][axis_id].pie(count_DF,explode=[0.05]*len(adt.obs[clus_names].unique()),pctdistance=-0.85,startangle=0,
#             labels=labels, colors=[morph_cMapDict[c] for c in keepOrder],
#           textprops = {"fontsize":12})
#     #plt.set_cmap('Set2')
#     #draw circle
#     centre_circle = plt.Circle((0,0),0.50,fc='white')
#     #fig = plt.gcf()
#     ax[0][axis_id].add_artist(centre_circle)
#     plt.tight_layout()
#     count_DF  = adt.obs[adt.obs.Parent=='White Matter'].sample(frac=1)[clus_names].value_counts()
#     keepOrder = count_DF.index
#     labels = []
#     for ix, x in count_DF.items():
#         if (100.*x/count_DF.sum())>=1:
#             labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
#         else:
#             if  "< 1%" not in labels:
#                 labels.append("< 1%")
#             else:
#                 labels.append("")
#     ax[1][axis_id].pie(count_DF.loc[keepOrder],explode=[0.05]*len(adt.obs[clus_names].unique()),pctdistance=-0.85,startangle=0,
#             labels=labels, colors=[morph_cMapDict[c] for c in keepOrder],
#           textprops = {"fontsize":12})
#     #draw circle
#     centre_circle = plt.Circle((0,0),0.50,fc='white')
#     #fig = plt.gcf()
#     ax[1][axis_id].add_artist(centre_circle)


def plotPie_custom(adt, ax, axis_id, cmap, feature):
    count_DF  = adt.obs[adataScaled2.obs.Parent=='Grey Matter'].sample(frac=1)[feature].value_counts()
    keepOrder = count_DF.index
    labels = []
    for ix, x in count_DF.items():
        print(100.*x/count_DF.sum())
        if (100.*x/count_DF.sum())>=1:
            labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
        else:
            if  "< 1%" not in labels:
                labels.append("< 1%")
            else:
                labels.append("")
    ax[0][axis_id].pie(count_DF,explode=None,pctdistance=-0.85,startangle=0,
            labels=labels, colors=[cmap[c] for c in keepOrder],
          textprops = {"fontsize":12})
    #plt.set_cmap('Set2')
    #draw circle
    centre_circle = plt.Circle((0,0),0.50,fc='white')
    #fig = plt.gcf()
    ax[0][axis_id].add_artist(centre_circle)
    plt.tight_layout()
    count_DF  = adt.obs[adt.obs.Parent=='White Matter'].sample(frac=1)[feature].value_counts()
    keepOrder = count_DF.index
    labels = []
    for ix, x in count_DF.items():
        if (100.*x/count_DF.sum())>=1:
            labels.append("{:.2f}%".format(100.*x/count_DF.sum()))
        else:
            if  "< 1%" not in labels:
                labels.append("< 1%")
            else:
                labels.append("")
    ax[1][axis_id].pie(count_DF.loc[keepOrder],explode=None,pctdistance=-0.85,startangle=0,
            labels=labels, colors=[cmap[c] for c in keepOrder],
          textprops = {"fontsize":12})
    #draw circle
    centre_circle = plt.Circle((0,0),0.50,fc='white')
    #fig = plt.gcf()
    ax[1][axis_id].add_artist(centre_circle)
    print('')
    

In [ ]:



import matplotlib.pyplot as plt
fig,ax=plt.subplots(2,3,figsize=(16,8))
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageType=='AD'].copy()
plotPie_custom(adataScaled2, ax, 0, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][0].set_title('AD - Grey Matter')
ax[1][0].set_title('AD - White Matter')
plt.tight_layout()

# exclude Ctl-2 sample, which corresponds to a patient with AML
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageID.isin(['Ctl-1','Ctl-3','Ctl-4'])].copy()
plotPie_custom(adataScaled2, ax, 1, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][1].set_title('Ctl - Grey Matter, AML excluded')
ax[1][1].set_title('Ctl - White Matter, AML excluded')

# plot for all control samples
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageType=='Ctl'].copy()
plotPie_custom(adataScaled2, ax, 2, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][2].set_title('Ctl - Grey Matter, AML included')
ax[1][2].set_title('Ctl - White Matter, AML included')
plt.tight_layout()

plt.savefig('figures/morph_figures/pieplots.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/pieplots.svg', format = 'svg')

# fdjkfld

# import matplotlib.pyplot as plt
# fig,ax=plt.subplots(2,3,figsize=(16,8))
# adataScaled2 = adataScaled[adataScaled.obs.ImageType=='AD'].copy()
# plotPie_custom(adataScaled2, ax, 0)
# ax[0][0].set_title('AD - Grey Matter')
# ax[1][0].set_title('AD - White Matter')
# plt.tight_layout()

# # exclude Ctl-2 sample, which corresponds to a patient with AML
# adataScaled2 = adataScaled[adataScaled.obs.ImageID.isin(['Ctl-1','Ctl-3','Ctl-4'])].copy()
# plotPie_custom(adataScaled2, ax, 1)
# ax[0][1].set_title('Ctl - Grey Matter, AML excluded')
# ax[1][1].set_title('Ctl - White Matter, AML excluded')

# # plot for all control samples
# adataScaled2 = adataScaled[adataScaled.obs.ImageType=='Ctl'].copy()
# plotPie_custom(adataScaled2, ax, 2)
# ax[0][2].set_title('Ctl - Grey Matter, AML included')
# ax[1][2].set_title('Ctl - White Matter, AML included')
# plt.tight_layout()
# plt.savefig('figures/pieplots.tiff')
# plt.savefig('figures/pieplots.svg', format = 'svg')



In [ ]:

import matplotlib.pyplot as plt
fig,ax=plt.subplots(2,4,figsize=(16,8))
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageType=='AD'].copy()
plotPie_custom(adataScaled2, ax, 0, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][0].set_title('AD - Grey Matter')
ax[1][0].set_title('AD - White Matter')
plt.tight_layout()

# exclude Ctl-2 sample, which corresponds to a patient with AML
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageID.isin(['Ctl-1','Ctl-3','Ctl-4'])].copy()
plotPie_custom(adataScaled2, ax, 1, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][1].set_title('Ctl - Grey Matter, AML excluded')
ax[1][1].set_title('Ctl - White Matter, AML excluded')

# plot for all control samples
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageType=='Ctl'].copy()
plotPie_custom(adataScaled2, ax, 2, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][2].set_title('Ctl - Grey Matter, AML included')
ax[1][2].set_title('Ctl - White Matter, AML included')

# plot for all control samples
adataScaled2 = morph_data_orig[morph_data_orig.obs.ImageID=='Ctl-2'].copy()
plotPie_custom(adataScaled2, ax, 3, cMapDict_morph, 'final_leiden_M1_updated')
ax[0][3].set_title('Ctl - Grey Matter, AML only')
ax[1][3].set_title('Ctl - White Matter, AML only')

plt.tight_layout()
plt.savefig('figures/morph_figures/pieplots_AML.tiff')
plt.savefig('figures/morph_figures/pieplots_AML.svg', format = 'svg')





In [ ]:
# stats

# compare abundances between GM and WM

import copy

ad_control_dict = {'2997': 'AD',
 '3155': 'AD',
 '3196': 'AD',
 '3280': 'AD',
 '1796': 'Ctl',
 '3628': 'Ctl',
 '3026': 'Ctl', 
 '1873': 'Ctl'}



diseases = []
regions = []
samples_list = []

prop_Rounded = []
prop_Intermediate = []
prop_Ramified_1 = []
prop_Ramified_2 = []
prop_Ramified_3 = []

counts_Rounded = []
counts_Intermediate = []
counts_Ramified_1 = []
counts_Ramified_2 = []
counts_Ramified_3 = []




samples = morph_data_orig.obs.ImageIDOLD.unique()

for sample in samples:
    
    sample_disease = ad_control_dict[str(sample)]
    
    sample_sub_GM = morph_data_orig[(morph_data_orig.obs.ImageIDOLD == sample) & (morph_data_orig.obs.Parent == 'Grey Matter')]
    counts_GM = sample_sub_GM.obs.final_leiden_M1_updated.value_counts()
    counts_GM_normalized = counts_GM/counts_GM.sum()
    
    diseases.append(sample_disease)
    regions.append('GM')
    samples_list.append(sample)

    
    
    prop_Rounded.append(counts_GM_normalized['Rounded'])
    prop_Intermediate.append(counts_GM_normalized['Intermediate'])
    prop_Ramified_1.append(counts_GM_normalized['Ramified 1'])
    prop_Ramified_2.append(counts_GM_normalized['Ramified 2'])
    prop_Ramified_3.append(counts_GM_normalized['Ramified 3'])
    
    
    
    counts_Rounded.append(counts_GM['Rounded'])
    counts_Intermediate.append(counts_GM['Intermediate'])
    counts_Ramified_1.append(counts_GM['Ramified 1'])
    counts_Ramified_2.append(counts_GM['Ramified 2'])
    counts_Ramified_3.append(counts_GM['Ramified 3'])
    
    
    
    sample_sub_WM = morph_data_orig[(morph_data_orig.obs.ImageIDOLD == sample) & (morph_data_orig.obs.Parent == 'White Matter')]
    counts_WM = sample_sub_WM.obs.final_leiden_M1_updated.value_counts()
    counts_WM_normalized = counts_WM/counts_WM.sum()
    
    diseases.append(sample_disease)
    regions.append('WM')
    samples_list.append(sample)

    
    
    prop_Rounded.append(counts_WM_normalized['Rounded'])
    prop_Intermediate.append(counts_WM_normalized['Intermediate'])
    prop_Ramified_1.append(counts_WM_normalized['Ramified 1'])
    prop_Ramified_2.append(counts_WM_normalized['Ramified 2'])
    prop_Ramified_3.append(counts_WM_normalized['Ramified 3'])
    
    
    
    counts_Rounded.append(counts_WM['Rounded'])
    counts_Intermediate.append(counts_WM['Intermediate'])
    counts_Ramified_1.append(counts_WM['Ramified 1'])
    counts_Ramified_2.append(counts_WM['Ramified 2'])
    counts_Ramified_3.append(counts_WM['Ramified 3'])
    
    
    
    
prop_df = pd.DataFrame({'disease': diseases, 'region': regions, 'Rounded':prop_Rounded,'Intermediate':prop_Intermediate, 'Ramified 1':prop_Ramified_1, 'Ramified 2':prop_Ramified_2, 'Ramified 3':prop_Ramified_3, 'sample':samples_list})
counts_df = pd.DataFrame({'disease': diseases, 'region': regions, 'Rounded':counts_Rounded,'Intermediate':counts_Intermediate, 'Ramified 1':counts_Ramified_1, 'Ramified 2':counts_Ramified_2, 'Ramified 3':counts_Ramified_3, 'sample':samples_list})

area_file_path = 'sample_area_files/'

areas = []
for row in counts_df.index:
    rowvals = counts_df.loc[row]
    sam = rowvals['sample']
    reg = rowvals['region']
    
    area_table = pd.read_csv(area_file_path + sam + '.csv')
    
    if reg == 'GM':
        ar = area_table[area_table['zone'] == 'Grey Matter']['area_mm'].iloc[0]
    elif reg == 'WM':
        ar = area_table[area_table['zone'] == 'White Matter']['area_mm'].iloc[0]
    areas.append(ar)
    


cells_per_mm2_df = copy.deepcopy(counts_df)
cells_per_mm2_df['areas'] = areas

for celltype in set(morph_data_orig.obs.final_leiden_M1_updated):
    cells_per_mm2_df[celltype] = cells_per_mm2_df[celltype]/cells_per_mm2_df['areas']


# count_DF  = morph_data_orig.obs[morph_data_orig.obs.Parent=='Grey Matter'].sample(frac=1).final_leiden_M1_updated.value_counts()
# count_DF/count_DF.sum()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95, savepath = False):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(7, 5))
    plt.suptitle(title, fontsize=10)  # Title for the whole plot

    # Bar plot
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('Proportion')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    # plt.subplot(1, 2, 2)
    # plt.boxplot([data1, data2], labels=[data1label, data2label], )
    # plt.ylabel('Proportion')
    # plt.title('Box and Whisker Plot')

    # Annotations
    
    logfc = np.log2(mean1/mean2)
    

    loc = np.mean([np.max(data1.append(data2)), 0])
    adj = 0.1 * (np.max(data1.append(data2)))
    plt.text(3, loc + adj, annotation1, fontsize=10, ha='center')
    plt.text(3, loc - adj, 'log2fc: ' + str(float("{:.3e}".format(logfc))), fontsize=10, ha='center')

    plt.tight_layout()
    
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()


# some t tests, between ad and control of GM
print('some t tests, between ad and control of GM')
from scipy.stats import ttest_ind

cell_types = morph_data_orig.obs.final_leiden_M1_updated.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = prop_df[prop_df.region == 'GM']
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of GM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    

    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM
print('')
print('some t tests, between ad and control of WM')

print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = prop_df[prop_df.region == 'WM']
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of WM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    

print('some t tests, between ad and control of GM AML EXCLUDED')
from scipy.stats import ttest_ind

cell_types = morph_data_orig.obs.final_leiden_M1_updated.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = prop_df[(prop_df.region == 'GM') & (prop_df['sample'] != '3628')]
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    

    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM
print('')
print('some t tests, between ad and control of WM AML EXCLUDED')

print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = prop_df[(prop_df.region == 'WM') & (prop_df['sample'] != '3628')]
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
        
    
    


    
    
    
    
    
# within AD, test between GM and WM

print('')
print('within AD, test between GM and WM')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = prop_df[prop_df.disease == 'AD']
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('within Ctl, test between GM and WM')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = prop_df[prop_df.disease == 'Ctl']
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of Ctl ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within AD, test between GM and WM

print('')
print('within AD, test between GM and WM AML excluded')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = prop_df[(prop_df.disease == 'AD') & (prop_df['sample'] != '3628')]
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('within Ctl, test between GM and WM AML excluded')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = prop_df[(prop_df.disease == 'Ctl') & (prop_df['sample'] != '3628')]
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of Ctl AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
# both ad and ctl, test between GM and WM

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    all_samps = copy.deepcopy(prop_df)
    
    
    GM_props = all_samps[all_samps.region == 'GM'][cell_type]    
    WM_props = all_samps[all_samps.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of all samples ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
# both ad and ctl, test between GM and WM AML exlcuded

print('')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    all_samps = prop_df[(prop_df['sample'] != '3628')]
    
    GM_props = all_samps[all_samps.region == 'GM'][cell_type]    
    WM_props = all_samps[all_samps.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    
    title = 't test, between GM and WM of all samples AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + '.svg')
    
    print(t_statistic)
    print(p_value)
    print('')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95, savepath = False):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(7, 5))
    plt.suptitle(title, fontsize=10)  # Title for the whole plot

    # Bar plot
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('Proportion')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    # plt.subplot(1, 2, 2)
    # plt.boxplot([data1, data2], labels=[data1label, data2label], )
    # plt.ylabel('Proportion')
    # plt.title('Box and Whisker Plot')

    # Annotations
    
    logfc = np.log2(mean1/mean2)
    

    loc = np.mean([np.max(data1.append(data2)), 0])
    adj = 0.1 * (np.max(data1.append(data2)))
    plt.text(3, loc + adj, annotation1, fontsize=10, ha='center')
    plt.text(3, loc - adj, 'log2fc: ' + str(float("{:.3e}".format(logfc))), fontsize=10, ha='center')

    plt.tight_layout()
    
    
    if savepath:
        plt.savefig(savepath)
    else:
        plt.show()

# some t tests, between ad and control of GM
print('some t tests, between ad and control of GM')
from scipy.stats import ttest_ind

cell_types = morph_data_orig.obs.final_leiden_M1_updated.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = cells_per_mm2_df[cells_per_mm2_df.region == 'GM']
    
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of GM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    

    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM
print('')
print('some t tests, between ad and control of WM')

print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = cells_per_mm2_df[cells_per_mm2_df.region == 'WM']
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of WM ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    

print('some t tests, between ad and control of GM AML EXCLUDED')
from scipy.stats import ttest_ind

cell_types = morph_data_orig.obs.final_leiden_M1_updated.unique()

alpha = 0.05 / len(cell_types)

print('Alpha:')
print(alpha)
print('')

print('GM')

for cell_type in cell_types:
    print(cell_type)
    
    GM_sub = cells_per_mm2_df[(cells_per_mm2_df.region == 'GM') & (cells_per_mm2_df['sample'] != '3628')]
    AD_props = GM_sub[GM_sub.disease == 'AD'][cell_type]    
    Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_type]
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    
    title = 't test, between ad and control of GM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    

    
    print(t_statistic)
    print(p_value)
    print('')

    
# some t tests, between ad and control of WM
print('')
print('some t tests, between ad and control of WM AML EXCLUDED')

print('WM')
    
for cell_type in cell_types:
    print(cell_type)
    
    WM_sub = cells_per_mm2_df[(cells_per_mm2_df.region == 'WM') & (cells_per_mm2_df['sample'] != '3628')]
    
    AD_props = WM_sub[WM_sub.disease == 'AD'][cell_type]    
    Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_type]
    
    
    
    
    t_statistic, p_value = ttest_ind(AD_props, Ctl_props)
    
    title = 't test, between ad and control of WM AML excluded ' + cell_type
    plot_data_comparison(AD_props, Ctl_props, 'AD', 'Ctl', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
        
    
    


    
    
    
    
    
# within AD, test between GM and WM

print('')
print('within AD, test between GM and WM')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = cells_per_mm2_df[cells_per_mm2_df.disease == 'AD']
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('within Ctl, test between GM and WM')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = cells_per_mm2_df[cells_per_mm2_df.disease == 'Ctl']
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of Ctl ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within AD, test between GM and WM

print('')
print('within AD, test between GM and WM AML excluded')
print('AD')
    
for cell_type in cell_types:
    print(cell_type)
    
    AD_sub = cells_per_mm2_df[(cells_per_mm2_df.disease == 'AD') & (cells_per_mm2_df['sample'] != '3628')]
    
    GM_props = AD_sub[AD_sub.region == 'GM'][cell_type]    
    WM_props = AD_sub[AD_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of AD AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')
    
    
    
    
    
# within Ctl, test between GM and WM

print('')
print('within Ctl, test between GM and WM AML excluded')
print('Ctl')
    
for cell_type in cell_types:
    print(cell_type)
    
    Ctl_sub = cells_per_mm2_df[(cells_per_mm2_df.disease == 'Ctl') & (cells_per_mm2_df['sample'] != '3628')]
    
    GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_type]    
    WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_type]
    
    t_statistic, p_value = ttest_ind(GM_props, WM_props)
    
    title = 't test, between GM and WM of Ctl AML excluded ' + cell_type
    plot_data_comparison(GM_props, WM_props, 'GM', 'WM', 'p val: ' + str(float("{:.3e}".format(p_value))), 't statistic: ' + str(float("{:.3e}".format(t_statistic))), 
                         title, savepath = 'figures/morph_figures/' + title + 'cells per mm2.svg')
    
    print(t_statistic)
    print(p_value)
    print('')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_data_comparison(data1, data2, data1label, data2label, annotation1, annotation2, title, ci_level=0.95):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(data1), np.mean(data2)
    std1, std2 = np.std(data1), np.std(data2)

    # Calculate confidence intervals (assuming normal distribution)
    ci_z_score = 1.96  # For 95% confidence interval

    ci1 = ci_z_score * (std1 / np.sqrt(len(data1)))
    ci2 = ci_z_score * (std2 / np.sqrt(len(data2)))

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.suptitle(title, fontsize=16)  # Title for the whole plot

    # Bar plot
    plt.subplot(1, 2, 1)
    plt.bar([1, 2], [mean1, mean2], yerr=[ci1, ci2], tick_label=[data1label, data2label], capsize=5)
    plt.scatter(np.ones_like(data1), data1, color='blue', alpha=0.5, label='Data 1')
    plt.scatter(2 * np.ones_like(data2), data2, color='orange', alpha=0.5, label='Data 2')
    plt.ylabel('Proportion')
    plt.title('Comparison of Data with Confidence Intervals')
    # plt.legend()
    # plt.legend(loc=)

    # Box and whisker plot
    plt.subplot(1, 2, 2)
    plt.boxplot([data1, data2], labels=[data1label, data2label], )
    plt.ylabel('Proportion')
    plt.title('Box and Whisker Plot')

    # Annotations
    plt.text(3, np.mean(data1), annotation1, fontsize=10, ha='center')
    plt.text(3, np.mean(data2), annotation2, fontsize=10, ha='center')

    plt.tight_layout()
    plt.show()

# Example usage
data1 = np.random.normal(loc=10, scale=2, size=100)  # Sample data 1
data2 = np.random.normal(loc=12, scale=2, size=100)  # Sample data 2
annotation1 = "Annotation 1"
annotation2 = "Annotation 2"
title = "Comparison of Data Sets"

plot_data_comparison(data1, data2, 'AD', 'CTL', annotation1, annotation2, title)





In [ ]:
# from scipy.stats import chi2_contingency
# import copy


# # chi squared

# # is there a significant relationship between AD and cell type proportions within GM?



# def calculate_standardized_residuals(observed, expected):
#     row_totals = np.sum(observed, axis=1)
#     column_totals = np.sum(observed, axis=0)
#     grand_total = np.sum(observed)
    
#     standardized_residuals = (observed - expected) / np.sqrt(expected * (1 - row_totals[:, np.newaxis] / grand_total) * (1 - column_totals / grand_total))
    
#     return standardized_residuals







# print('GM')

# prop_df_hundred = prop_df[cell_types]*100
# prop_df_hundred['disease'] = prop_df['disease']
# prop_df_hundred['region'] = prop_df['region']

# #counts_df = prop_df_hundred

    
# GM_sub = counts_df[counts_df.region == 'GM']

# AD_props = GM_sub[GM_sub.disease == 'AD'][cell_types].mean()
# Ctl_props = GM_sub[GM_sub.disease == 'Ctl'][cell_types].mean()

# observed = [AD_props, Ctl_props]

# chi2, p, dof, expected = chi2_contingency(observed)

# print("Chi-squared statistic:", chi2)
# print("p-value:", p)
# print("Degrees of freedom:", dof)
# print("Expected frequencies:")
# print(expected)

# residuals = calculate_standardized_residuals(observed, expected)
# significant_deviations = np.where(np.abs(residuals) > 2)

# resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['AD'] * 5) + (['Ctl'] * 5)})

# print(resid_df)

# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
# sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
# plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
# plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
# plt.title('GM ad vs control')
# plt.xlabel('Cell Types')
# plt.ylabel('Residuals')
# plt.legend(loc='upper left')
# plt.show()




# print('WM')


    
# WM_sub = counts_df[counts_df.region == 'WM']

# AD_props = WM_sub[WM_sub.disease == 'AD'][cell_types].mean()
# Ctl_props = WM_sub[WM_sub.disease == 'Ctl'][cell_types].mean()

# observed = [AD_props, Ctl_props]

# chi2, p, dof, expected = chi2_contingency(observed)

# print("Chi-squared statistic:", chi2)
# print("p-value:", p)
# print("Degrees of freedom:", dof)
# print("Expected frequencies:")
# print(expected)

# residuals = calculate_standardized_residuals(observed, expected)
# significant_deviations = np.where(np.abs(residuals) > 2)

# resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['AD'] * 5) + (['Ctl'] * 5)})
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
# sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
# plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
# plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
# plt.title('WM ad vs control')
# plt.xlabel('Cell Types')
# plt.ylabel('Residuals')
# plt.legend(loc='upper left')
# plt.show()





# print('AD')


    
# AD_sub = counts_df[counts_df.disease == 'AD']

# GM_props = AD_sub[AD_sub.region == 'GM'][cell_types].mean()
# WM_props = AD_sub[AD_sub.region == 'WM'][cell_types].mean()

# observed = [GM_props, WM_props]

# chi2, p, dof, expected = chi2_contingency(observed)

# print("Chi-squared statistic:", chi2)
# print("p-value:", p)
# print("Degrees of freedom:", dof)
# print("Expected frequencies:")
# print(expected)

# residuals = calculate_standardized_residuals(observed, expected)
# significant_deviations = np.where(np.abs(residuals) > 2)

# resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['GM'] * 5) + (['WM'] * 5)})
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
# sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
# plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
# plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
# plt.title('AD gm vs wm')
# plt.xlabel('Cell Types')
# plt.ylabel('Residuals')
# plt.legend(loc='upper left')
# plt.show()



# print('Ctl')


    
# Ctl_sub = counts_df[counts_df.disease == 'Ctl']

# GM_props = Ctl_sub[Ctl_sub.region == 'GM'][cell_types].mean()
# WM_props = Ctl_sub[Ctl_sub.region == 'WM'][cell_types].mean()

# observed = [GM_props, WM_props]

# chi2, p, dof, expected = chi2_contingency(observed)

# print("Chi-squared statistic:", chi2)
# print("p-value:", p)
# print("Degrees of freedom:", dof)
# print("Expected frequencies:")
# print(expected)

# residuals = calculate_standardized_residuals(observed, expected)
# significant_deviations = np.where(np.abs(residuals) > 2)

# resid_df = pd.DataFrame({'values':list(residuals[0]) + list(residuals[1]), 'cell_types':list(cell_types) + list(cell_types), 'cat':(['GM'] * 5) + (['WM'] * 5)})
# sns.set(style="whitegrid")
# plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
# sns.barplot(data=resid_df, x='cell_types', y='values', hue='cat')
# plt.axhline(y=-2, color='r', linestyle='--', linewidth=1)
# plt.axhline(y=2, color='r', linestyle='--', linewidth=1)
# plt.title('Ctl gm vs wm')
# plt.xlabel('Cell Types')
# plt.ylabel('Residuals')
# plt.legend(loc='upper left')
# plt.show()



In [ ]:
## Plot AD statistics


# dt = 'distance_to_nearest_aBeta'
# ss = morph_data[(morph_data.obs.ImageType =='AD')]
# ss = ss[ss.obs.Parent=='Grey Matter']
# BINS  =[-1,0,5,10,15,20,30,50,75,100,200,500]
# out = pd.cut(ss.obs[dt],bins=BINS, right=True)
# ss.obs['Quantile'] = out.values
 

In [ ]:


# fig,ax = plt.subplots(2,1,figsize=(8,8),gridspec_kw={'height_ratios': [1.6,1.2]})

# matpl=sc.pl.matrixplot(ss, var_names = morph_data_orig.var_names,groupby='Quantile',show=False,vmin=-0.5, vmax=0.75,vcenter=0,
#                  dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,return_fig=True,swap_axes=True)
# matpl.legend(show=False)

# import numpy as np
# sns.set_style("whitegrid", {'axes.grid' : False})


# oDF = ss.obs.groupby(['Quantile','final_leiden_M1','ImageID']).size().T.reset_index().groupby(['Quantile','final_leiden_M1']).mean().unstack(level=1).T.reset_index(drop=True)
# oDFPct = 100*oDF/oDF.sum(axis=0)

# oDFPct.T.plot(kind='bar',stacked=True,ax=ax[0],color=morph_cMapDict, legend=True, )
# ax[0].legend(title='Cluster ID',loc=(1,0.0))
# ax[0].set_ylabel('Percentage of \n Microglial cells')
# ax[0].set_xlabel('')
# ax[0].set_xticks([])



# # find MG dections that are clusters
# oDF =100.*ss.obs[ss.obs['numCellCentroids']>1].groupby(['Quantile'])[['numCellCentroids']].size()/ss.obs.groupby(['Quantile'])[['numCellCentroids']].size()

# oDFPct = oDF.T#/oDF.sum(axis=1)
# oDFPct.T.plot(kind='bar',stacked=True,ax=ax[1], legend=False, color='skyblue')
# ax[1].set_ylabel('% of cells that \n are Microglial clusters')
# ax[1].set_xlabel('Grouped distance to the nearest Plaque (µm)')
# ax[1].set_xticklabels(['[0]']+ax[1].get_xticklabels()[1:],rotation = 45)
# ax[1].set_ylim(0,40)

# plt.tight_layout()
# plt.savefig('pixel_clustering_newmethod/paper_images/morph_images/morph_distance_analysis.tiff')



In [ ]:
# morphology vs marker expression


matrix1 = morph_data_matrix
matrix2 = protein_data_matrix


correlation_matrix = pd.DataFrame()

# Calculate correlations for all pairs of features
for col1 in matrix1.columns:
    for col2 in matrix2.columns:
        correlation = matrix1[col1].corr(matrix2[col2])
        correlation_matrix.at[col1, col2] = correlation

# Print the correlation matrix
print(correlation_matrix)


plt.figure(figsize=(14, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Morphology x Protein correlation')
plt.tight_layout()
plt.savefig('figures/morph_figures/morph_vs_protein_corr.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_protein_corr.svg', format = 'svg')

plt.show()

In [ ]:
# morphology vs marker expression

ad_cells = morph_data_orig[morph_data_orig.obs['ImageType'] == 'AD'].obs['ImageID_CellID']


matrix1 = morph_data_matrix.loc[ad_cells]
matrix2 = protein_data_matrix.loc[ad_cells]


correlation_matrix = pd.DataFrame()

# Calculate correlations for all pairs of features
for col1 in matrix1.columns:
    for col2 in matrix2.columns:
        correlation = matrix1[col1].corr(matrix2[col2])
        correlation_matrix.at[col1, col2] = correlation

# Print the correlation matrix
print(correlation_matrix)


plt.figure(figsize=(14, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Morphology x Protein correlation AD')
plt.tight_layout()
plt.savefig('figures/morph_figures/morph_vs_protein_corr_AD.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_protein_corr_AD.svg', format = 'svg')

plt.show()




In [ ]:
# morphology vs marker expression



control_cells = morph_data_orig[morph_data_orig.obs['ImageType'] == 'Ctl'].obs['ImageID_CellID']


matrix1 = morph_data_matrix.loc[control_cells]
matrix2 = protein_data_matrix.loc[control_cells]

correlation_matrix = pd.DataFrame()

# Calculate correlations for all pairs of features
for col1 in matrix1.columns:
    for col2 in matrix2.columns:
        correlation = matrix1[col1].corr(matrix2[col2])
        correlation_matrix.at[col1, col2] = correlation

# Print the correlation matrix
print(correlation_matrix)


plt.figure(figsize=(14, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Morphology x Protein correlation Ctl')
plt.tight_layout()
plt.savefig('figures/morph_figures/morph_vs_protein_corr_Ctl.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_protein_corr_Ctl.svg', format = 'svg')

plt.show()




In [ ]:

# morphology vs leiden

# cMapDict = {0: '#1f77b4',
#  1: '#ff7f0e',
#  2: '#279e68',
#  3: '#d62728',
#  4: '#aa40fc',
#  5: '#8c564b',
#  6: '#e377c2',
#  7: '#b5bd61',
#  8: '#17becf',
#  9: '#aec7e8',
#  10: '#ffbb78'}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': morph_data_orig.obs['final_leiden_M1_updated'],
    'Category2': morph_data_orig.obs['protein_leiden_updated']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Morphology clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Expr. clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.tight_layout()

plt.savefig('figures/morph_figures/morph_vs_leiden_barchart.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_leiden_barchart.svg', format = 'svg')


plt.show()















In [ ]:
# morphology vs leiden

# cMapDict = {0: '#1f77b4',
#  1: '#ff7f0e',
#  2: '#279e68',
#  3: '#d62728',
#  4: '#aa40fc',
#  5: '#8c564b',
#  6: '#e377c2',
#  7: '#b5bd61',
#  8: '#17becf',
#  9: '#aec7e8',
#  10: '#ffbb78'}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': morph_data_orig.obs['protein_leiden_updated'],
    'Category2': morph_data_orig.obs['final_leiden_M1_updated']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.tight_layout()

plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted.svg', format = 'svg')

plt.show()

In [ ]:
morph_data_orig.obs

In [ ]:
# split above by ad and control

# AD

# morphology vs leiden

# cMapDict = {0: '#1f77b4',
#  1: '#ff7f0e',
#  2: '#279e68',
#  3: '#d62728',
#  4: '#aa40fc',
#  5: '#8c564b',
#  6: '#e377c2',
#  7: '#b5bd61',
#  8: '#17becf',
#  9: '#aec7e8',
#  10: '#ffbb78'}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': morph_data_orig[morph_data_orig.obs.ImageType == 'AD'].obs['protein_leiden_updated'],
    'Category2': morph_data_orig[morph_data_orig.obs.ImageType == 'AD'].obs['final_leiden_M1_updated']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering AD')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.tight_layout()

plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted_AD.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted_AD.svg', format = 'svg')

plt.show()




# Control

# morphology vs leiden

# cMapDict = {0: '#1f77b4',
#  1: '#ff7f0e',
#  2: '#279e68',
#  3: '#d62728',
#  4: '#aa40fc',
#  5: '#8c564b',
#  6: '#e377c2',
#  7: '#b5bd61',
#  8: '#17becf',
#  9: '#aec7e8',
#  10: '#ffbb78'}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': morph_data_orig[morph_data_orig.obs.ImageType == 'Ctl'].obs['protein_leiden_updated'],
    'Category2': morph_data_orig[morph_data_orig.obs.ImageType == 'Ctl'].obs['final_leiden_M1_updated']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering control')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.tight_layout()

plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted_Ctl.tiff', format = 'tiff')
plt.savefig('figures/morph_figures/morph_vs_leiden_barchart_inverted_Ctl.svg', format = 'svg')

plt.show()